# NHTSA Vehicle Safety Analysis
### Group 9 Final Project — BUSN 32120

---

## Introduction

In the United States, over 40,000 people die in motor vehicle crashes every year. Behind every fatality statistic lies a chain of preventable failures — design flaws missed in testing, defects ignored until they reach critical mass, and recalls issued too late to prevent harm.

This project uses data from the **National Highway Traffic Safety Administration (NHTSA)** to trace that chain end-to-end. We follow a vehicle safety defect through its entire lifecycle:

> **Crash Test Ratings** → **Consumer Complaints** → **Federal Investigations** → **Recall Actions** → **Predictive Models**

The central question: *Can publicly available safety data help consumers identify dangerous vehicles before they become tomorrow's headline?*

---

## Data Sources & Methods

| Dataset | Description | Key Fields |
|---------|-------------|------------|
| **Safety Ratings** | NCAP crash test scores (1–5 stars) | Overall, Frontal, Side, Rollover stars |
| **Complaints** | Consumer-reported defects filed with NHTSA | Make, Model, Component, Injuries, Deaths |
| **Investigations** | Formal defect investigations opened by NHTSA | Subject, Status, Related Campaigns |
| **Recalls** | Manufacturer-issued safety recall campaigns | Affected Vehicles, Component, Severity |

**SQL Techniques Used:**  
JOINs | Window Functions (LAG, RANK, PERCENT_RANK, SUM OVER) | GROUP BY CUBE & ROLLUP | CTEs | Correlated Subqueries | EXISTS | CASE WHEN | HAVING with Dynamic Benchmarks

**Predictive Models:**  
Linear Regression (injury prediction) | Logistic Regression (recall prediction)

---

## About NHTSA

The **National Highway Traffic Safety Administration** is the federal agency responsible for motor vehicle safety in the United States. Operating within the Department of Transportation, NHTSA:

- Sets and enforces **Federal Motor Vehicle Safety Standards (FMVSS)**
- Conducts independent **crash tests** through the New Car Assessment Program (NCAP)
- Investigates reported defects and compels **recalls** when warranted
- Maintains public databases of complaints, investigations, and recall campaigns

NCAP evaluates vehicles across three dimensions:

1. **Occupant protection** — survivability in frontal, side, and rollover collisions
2. **Rollover resistance** — stability under extreme maneuvers (critical for SUVs and trucks)
3. **Crash avoidance** — effectiveness of ADAS features like automatic emergency braking and lane-keeping

Manufacturers are legally required to report safety defects within five business days (49 CFR Part 573) and submit ongoing recall completion data to NHTSA's Office of Defects Investigation.

---

## Motivation

A 5-star crash test rating tells you a vehicle protects occupants well *at the moment of sale*. It says nothing about:

- Whether the transmission will fail at 40,000 miles
- Whether a design flaw will trigger a recall affecting millions of vehicles
- Whether the manufacturer has a pattern of delaying corrective action

This analysis connects **laboratory safety scores with real-world outcomes** to answer questions that no single dataset can address alone:

- Do higher-rated vehicles actually experience fewer safety issues over their lifetime?
- Which brands show recurring defect patterns despite strong crash test results?
- Can complaint volume predict when a major recall is coming?
- Does modern safety technology measurably reduce real-world harm?

---

## Target Audience

This analysis is built for **U.S. car buyers who treat safety as a non-negotiable purchasing criterion**.

Specifically, it helps consumers:

- Look beyond headline star ratings to evaluate long-term reliability
- Identify vehicles and brands with low complaint-to-recall pipelines
- Understand which safety technologies deliver measurable protection
- Recognize platform-sharing risks (one defect, multiple badges)

Secondary audiences: **auto insurers** (underwriting and risk modeling) and **fleet managers** (procurement liability).

---

## Notebook Structure

| Part | Topic | Key Question |
|------|-------|--------------|
| 1 | Safety Ratings | Are cars actually getting safer over time? |
| 2 | Consumer Complaints | Who gets complained about, and for what? |
| 3 | Complaints vs. Ratings | Do safer-rated cars get fewer complaints? |
| 4 | Investigations | When does NHTSA take formal action? |
| 5 | Recalls | What gets recalled, and how severe is it? |
| 6 | Hall of Shame | Which vehicles failed tests AND got recalled? |
| 7 | Predictive Models | Can we predict injuries and recalls? |

In [ ]:
# ================================================================
# Setup: Imports & Configuration
# ================================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

SVG_SAVE = True

if SVG_SAVE:
    os.makedirs('Plots', exist_ok=True)

---
# Part 1: Are Cars Actually Safe?
*Before looking at what goes wrong, let's establish what "safe" looks like. NHTSA crash-tests every major vehicle sold in America — how do body styles compare, and are ratings improving over time?*

## Query 1: Safety Ratings by Body Style

We start by establishing what "safe" looks like. This analysis examines how NHTSA safety ratings vary across different vehicle body styles (sedan, SUV, truck, etc.). By averaging overall, frontal, side, and rollover star ratings per body type, we identify which vehicle categories offer the best crash protection.

In [ ]:
%%sql -r SafetyRatingsByBodyStyle
-- ================================================================
-- Query 1: Safety Ratings by Body Style
-- ================================================================
-- Calculates average star ratings (overall, frontal, side, rollover)
-- grouped by vehicle body style to identify which body types
-- perform best/worst in NHTSA crash testing.
-- ================================================================

SELECT
    BODY_STYLE,
    COUNT(*) AS VEHICLES_RATED,
    ROUND(AVG(OVERALL_STARS), 2) AS AVG_OVERALL_STARS,
    ROUND(AVG(OVERALL_FRNT_STARS), 2) AS AVG_FRONTAL_STARS,
    ROUND(AVG(OVERALL_SIDE_STARS), 2) AS AVG_SIDE_STARS,
    ROUND(AVG(ROLLOVER_STARS), 2) AS AVG_ROLLOVER_STARS,
    ROUND(AVG(ROLLOVER_POSSIBILITY), 2) AS AVG_ROLLOVER_PROBABILITY
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY
WHERE OVERALL_STARS IS NOT NULL
GROUP BY BODY_STYLE
HAVING COUNT(*) >= 5
ORDER BY AVG_OVERALL_STARS DESC;

In [ ]:
# ================================================================
# Query 1 Visualization: Safety Ratings by Body Style
# ================================================================

# ----------------------------------------------------------------
# Data Cleaning: Consolidate duplicate/variant body style codes
# into canonical categories so the chart isn't cluttered with
# near-identical entries (e.g. '4DR', '4 DR', '4 DR Early Release'
# all become '4-Door Sedan')
# ----------------------------------------------------------------
body_style_consolidate = {
    '2 DR': '2-Door Coupe',
    '3 HB': 'Hatchback',
    '3C': 'Coupe',
    '4': '4-Door Sedan',
    '4 DR': '4-Door Sedan',
    '4 DR Early Release': '4-Door Sedan',
    '4DR': '4-Door Sedan',
    '4DR SUV': 'SUV',
    '5 HB': 'Hatchback',
    '5HB': 'Hatchback',
    'C': 'Coupe',
    'EXTENDED SUV': 'SUV',
    'HYBRID PU/CC': 'Pickup Truck',
    'MV': 'Minivan',
    'PU/CC': 'Pickup Truck',
    'PU/CC Early Release': 'Pickup Truck',
    'PU/CC Later Release': 'Pickup Truck',
    'PU/EC': 'Pickup Truck',
    'PU/RC': 'Pickup Truck',
    'PV': 'Passenger Van',
    'PV LATER RELEASE': 'Passenger Van',
    'SUV': 'SUV',
    'SUV EARLY RELEASE': 'SUV',
    'SUV Early Release': 'SUV',
    'SUV Later Release': 'SUV',
    'SW': 'Station Wagon',
    'VAN': 'Van',
    'WAGON': 'Station Wagon'
}
SafetyRatingsByBodyStyle['BODY_STYLE_CLEAN'] = SafetyRatingsByBodyStyle['BODY_STYLE'].map(body_style_consolidate).fillna(SafetyRatingsByBodyStyle['BODY_STYLE'])

# ----------------------------------------------------------------
# Re-aggregate: Group by the cleaned body style and recalculate
# weighted averages so consolidated categories reflect all vehicles
# ----------------------------------------------------------------
df_plot = SafetyRatingsByBodyStyle.groupby('BODY_STYLE_CLEAN').agg(
    VEHICLES_RATED=('VEHICLES_RATED', 'sum'),
    AVG_OVERALL_STARS=('AVG_OVERALL_STARS', 'mean'),
    AVG_FRONTAL_STARS=('AVG_FRONTAL_STARS', 'mean'),
    AVG_SIDE_STARS=('AVG_SIDE_STARS', 'mean'),
    AVG_ROLLOVER_STARS=('AVG_ROLLOVER_STARS', 'mean')
).sort_values('AVG_OVERALL_STARS', ascending=False).reset_index()

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 6))
categories = ['AVG_OVERALL_STARS', 'AVG_FRONTAL_STARS', 'AVG_SIDE_STARS', 'AVG_ROLLOVER_STARS']
colors = ['#e94560', '#f5a623', '#00d2d3', '#a29bfe']
labels = ['Overall', 'Frontal', 'Side', 'Rollover']
x = np.arange(len(df_plot))
width = 0.19

for i, (cat, c, lbl) in enumerate(zip(categories, colors, labels)):
    ax.bar(x + i * width, df_plot[cat], width, label=lbl, color=c, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_plot['BODY_STYLE_CLEAN'], rotation=40, ha='right', fontsize=9)
ax.set_ylim(0, 5.5)
ax.axhline(y=5, color='#ffd700', linestyle='--', alpha=0.4)
ax.set_xlabel('Body Style')
ax.set_ylabel('Avg Star Rating')
ax.set_title('Which Body Styles Survive Crashes Best?', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=9, ncol=4)
ax.grid(axis='y', alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 1 - Which Body Styles Survive Crashes Best.svg', format='svg', bbox_inches='tight')

### Findings: Safety Ratings by Body Style

- **Which Body Styles Survive Crashes Best?** — 4-door sedans lead overall star averages, benefiting from decades of crash optimization and the highest testing volume. SUVs perform nearly as well in frontal and side impacts, but their physics (higher center of gravity, greater weight) still penalize them in rollover resistance — despite major improvements from electronic stability control
- **Where's the Weak Spot?** — Rollover ratings show the most variation across body styles. Pickup trucks and vans score well in frontal and side impacts but still lag in rollover resistance — their height and weight distribution create physics that no amount of engineering fully overcomes
- **Is 5 Stars Achievable?** — The dashed gold reference line at 5 stars reveals the answer: almost no body style averages a perfect score across all categories. 4.5+ is the realistic ceiling for most vehicle types

## Query 2: Safety Rating Trends Year-over-Year (WINDOW FUNCTIONS)

Are cars getting safer over time? This query leverages multiple window functions — `LAG()`, `AVG() OVER ROWS`, `PERCENT_RANK()`, `SUM() OVER UNBOUNDED PRECEDING`, and `RANK()` — to perform a comprehensive time-series analysis of safety ratings, revealing rolling trends, cumulative achievements, and fleet-wide improvement patterns.

In [ ]:
%%sql -r SafetyRatingsYearOverYear
-- ================================================================
-- Query 2: Safety Rating Trends Year-over-Year (WINDOW FUNCTIONS)
-- ================================================================
-- Uses multiple window functions to analyze safety rating trends:
-- - LAG() for year-over-year change detection
-- - AVG() OVER ROWS for 3-year rolling average (smoothing)
-- - PERCENT_RANK() for relative safety positioning
-- - SUM() OVER UNBOUNDED PRECEDING for cumulative high ratings
-- - RANK() for within-year ranking
-- ================================================================

SELECT
    MAKE,
    MODEL,
    MODEL_YR,
    OVERALL_STARS,
    ROLLOVER_POSSIBILITY,
    PERCENT_RANK() OVER (ORDER BY OVERALL_STARS DESC) AS SAFETY_PERCENTILE,
    LAG(OVERALL_STARS) OVER (PARTITION BY MAKE ORDER BY MODEL_YR) AS PREV_YEAR_STARS,
    OVERALL_STARS - LAG(OVERALL_STARS) OVER (PARTITION BY MAKE ORDER BY MODEL_YR) AS STAR_CHANGE,
    AVG(OVERALL_STARS) OVER (PARTITION BY MAKE ORDER BY MODEL_YR ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ROLLING_3YR_AVG,
    RANK() OVER (PARTITION BY MODEL_YR ORDER BY OVERALL_STARS DESC) AS RANK_IN_YEAR,
    COUNT(*) OVER (PARTITION BY MAKE) AS TOTAL_MODELS_BY_MAKE,
    SUM(CASE WHEN OVERALL_STARS >= 4 THEN 1 ELSE 0 END) OVER (PARTITION BY MAKE ORDER BY MODEL_YR ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS CUMULATIVE_HIGH_RATINGS
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY
WHERE OVERALL_STARS IS NOT NULL
ORDER BY MAKE, MODEL_YR;

In [ ]:
# ================================================================
# Query 2 Visualization: Safety Rating Trends (Window Functions)
# ================================================================

for col in ['MODEL_YR', 'OVERALL_STARS', 'ROLLING_3YR_AVG', 'SAFETY_PERCENTILE', 'CUMULATIVE_HIGH_RATINGS', 'STAR_CHANGE', 'RANK_IN_YEAR']:
    SafetyRatingsYearOverYear[col] = pd.to_numeric(SafetyRatingsYearOverYear[col], errors='coerce')

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Safety Ratings — Window Function Analysis', fontsize=14, fontweight='bold')

ax = axes[0, 0]
df_change = SafetyRatingsYearOverYear.dropna(subset=['STAR_CHANGE']).copy()
df_change['DIRECTION'] = df_change['STAR_CHANGE'].apply(lambda x: 'Improved' if x > 0 else ('Declined' if x < 0 else 'No Change'))
change_pivot = df_change.groupby(['MODEL_YR', 'DIRECTION']).size().unstack(fill_value=0)
if 'Improved' in change_pivot.columns and 'Declined' in change_pivot.columns:
    change_pivot[['Improved', 'No Change', 'Declined']].plot.bar(
        stacked=True, ax=ax, color=['#55efc4', '#636e72', '#e94560'], width=0.8)
ax.set_title('LAG(): Are Cars Getting Safer?', fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('Count')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.15, linestyle='--')
ax.tick_params(axis='x', rotation=45, labelsize=7)

ax = axes[0, 1]
top5_makes = SafetyRatingsYearOverYear.groupby('MAKE')['OVERALL_STARS'].count().nlargest(5).index
colors_map = dict(zip(top5_makes, ['#e94560', '#f5a623', '#00d2d3', '#a29bfe', '#55efc4']))
for make in top5_makes:
    df_m = SafetyRatingsYearOverYear[SafetyRatingsYearOverYear['MAKE'] == make].dropna(subset=['ROLLING_3YR_AVG'])
    yearly = df_m.groupby('MODEL_YR')['ROLLING_3YR_AVG'].mean()
    ax.plot(yearly.index, yearly.values, label=make, color=colors_map[make], lw=2)
ax.set_title('AVG() OVER ROWS: Smoothed Safety Trend', fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('Rolling Avg Stars')
ax.legend(fontsize=8)
ax.grid(alpha=0.15, linestyle='--')

ax = axes[0, 2]
df_pct = SafetyRatingsYearOverYear.dropna(subset=['SAFETY_PERCENTILE', 'MODEL_YR']).copy()
df_pct['PERCENTILE_TIER'] = pd.cut(df_pct['SAFETY_PERCENTILE'], bins=[-.01, 0.25, 0.50, 0.75, 1.0], labels=['Top 25%', '25\u201350%', '50\u201375%', 'Bottom 25%'])
tier_pivot = df_pct.groupby(['MODEL_YR', 'PERCENTILE_TIER']).size().unstack(fill_value=0)
tier_pivot_pct = tier_pivot.div(tier_pivot.sum(axis=1), axis=0) * 100
tier_pivot_pct[['Top 25%', '25\u201350%', '50\u201375%', 'Bottom 25%']].plot.bar(
    stacked=True, ax=ax, color=['#00b894', '#fdcb6e', '#e17055', '#d63031'], width=0.8)
ax.set_title('PERCENT_RANK(): Safety Tiers Over Time: Is the Herd Getting Safer?', fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('% of Vehicles')
ax.legend(fontsize=8, loc='lower right')
ax.grid(axis='y', alpha=0.15, linestyle='--')
ax.tick_params(axis='x', rotation=45, labelsize=7)

ax = axes[1, 0]
for make in top5_makes:
    df_m = SafetyRatingsYearOverYear[SafetyRatingsYearOverYear['MAKE'] == make].sort_values('MODEL_YR')
    cumulative = df_m.groupby('MODEL_YR')['CUMULATIVE_HIGH_RATINGS'].max()
    ax.plot(cumulative.index, cumulative.values, label=make, color=colors_map[make], lw=2)
ax.set_title('SUM() OVER: Who Builds the Most Safe Cars?', fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('Cumulative 4+ Star Models')
ax.legend(fontsize=8)
ax.grid(alpha=0.15, linestyle='--')

ax = axes[1, 1]
rank1_counts = SafetyRatingsYearOverYear[SafetyRatingsYearOverYear['RANK_IN_YEAR'] == 1].groupby('MODEL_YR').size()
ax.bar(rank1_counts.index, rank1_counts.values, color='#ffd700', edgecolor='white', linewidth=0.3)
ax.set_title('RANK(): How Competitive Is the Top Spot?', fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('# of Rank-1 Vehicles')
ax.grid(axis='y', alpha=0.15, linestyle='--')

axes[1, 2].axis('off')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 2 - Safety Ratings Window Function Analysis.svg', format='svg', bbox_inches='tight')

### Findings: Safety Ratings Year-over-Year Trends (Window Functions)

- **Are Cars Getting Safer?** — The overwhelming answer is "they're staying the same." 79% of year-over-year transitions show no change, with improvements (11%) barely edging out declines (10%). The industry isn't regressing, but it's not leaping forward either — most vehicles lock in their rating and maintain it
- **Smoothed Safety Trend by Make** — The rolling average strips away noise to reveal the truth: all top 5 makes hover above 4.3 stars. Toyota edges everyone at 98.4% of models scoring 4+ stars. The lines are converging, meaning the gap between the best and worst major manufacturers is shrinking
- **Whe herd is getting safer?** — The fleet is converging toward the top. Early model years show an even spread across percentile tiers, but post-2015 the "Top 25%" band dominates the stack as safety tech became standard — the gap between good and great is narrowing, and 4–5 stars are now table stakes rather than exceptional
- **Who Builds the Most Safe Cars Over Time?** — Ford has the most 4+ star models in raw numbers (524), but that's partly because they test more vehicles. Toyota's curve is steeper relative to its fleet size — they're batting nearly perfect. Flat sections in any make's line reveal years where they coasted
- **How Competitive Is the Top Spot?** — More vehicles are tying for #1 every year. That's actually great news — it means 5-star safety is becoming table stakes rather than a differentiator. The bar chart rising over time shows the entire industry lifting its floor

---
# Part 2: What Do Consumers Report?
*Every safety defect starts with a driver picking up the phone. Who gets complained about, how are rankings shifting over time, and what components fail the most?*

## Query 3: Ranking Makes by Consumer Complaints

Now that we know what safe looks like, let's see what consumers actually report. This analysis ranks vehicle manufacturers by total NHTSA complaint volume, including death rates, injury counts, and crash/fire frequencies — separating volume effects (popular brands) from genuine safety outliers.

In [ ]:
%%sql -r RankingMakesByComplaints
-- ================================================================
-- Query 3: Ranking Makes by Consumer Complaints
-- ================================================================
-- Aggregates all consumer complaints by vehicle make to produce
-- a ranked leaderboard of the top 20 most-complained-about brands.
-- Includes death/injury totals and a normalized death rate per
-- complaint to distinguish volume from severity.
-- ================================================================

SELECT
    MAKETXT AS MAKE,
    COUNT(*) AS TOTAL_COMPLAINTS,
    SUM(DEATHS) AS TOTAL_DEATHS,
    SUM(INJURED) AS TOTAL_INJURIES,
    SUM(CASE WHEN CRASH = 'Y' THEN 1 ELSE 0 END) AS CRASH_COUNT,
    SUM(CASE WHEN FIRE = 'Y' THEN 1 ELSE 0 END) AS FIRE_COUNT,
    ROUND(SUM(DEATHS) * 100.0 / NULLIF(COUNT(*), 0), 4) AS DEATH_RATE_PER_COMPLAINT
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
GROUP BY MAKETXT
ORDER BY TOTAL_COMPLAINTS DESC
LIMIT 20;

In [ ]:
# ================================================================
# Query 3 Visualization: Who Gets Complained About the Most?
# ================================================================

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(16, 8))

colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(RankingMakesByComplaints)))

bars = ax.barh(
    RankingMakesByComplaints['MAKE'][::-1],
    RankingMakesByComplaints['TOTAL_COMPLAINTS'][::-1],
    color=colors,
    edgecolor='white',
    linewidth=0.3
)

ax.set_title('Who Gets Complained About the Most?', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Total Complaints', fontsize=12, fontweight='bold')
ax.bar_label(bars, fmt='{:,.0f}', padding=-42, fontsize=9, color='white', fontweight='bold')

for i, value in enumerate(RankingMakesByComplaints['DEATH_RATE_PER_COMPLAINT'][::-1]):
    ax.text(
        bars[i].get_width() + 250,
        bars[i].get_y() + bars[i].get_height()/2,
        f'Deaths/Complaint: {value:.4f}%',
        va='center', fontsize=8.5, color='#e94560', fontweight='bold'
    )

ax.grid(axis='x', linestyle='--', alpha=0.15)
ax.set_xlim(0, RankingMakesByComplaints['TOTAL_COMPLAINTS'].max() * 1.18)
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 3 - Who Gets Complained About the Most.svg', format='svg', bbox_inches='tight')

### Findings: Ranking Makes by Consumer Complaints

- **Who Gets Complained About the Most?** — Ford, Toyota, and Chevrolet dominate the leaderboard, consistent with their massive U.S. market shares. But volume alone doesn't indicate danger — it mostly reflects how many vehicles are on the road
- **Does High Volume Mean High Danger?** — Not necessarily. The death rate per complaint separates signal from noise: some mid-volume brands show elevated fatality rates that a raw count ranking would hide. A brand at position #10 by volume might be more dangerous per incident than the #1 spot

## Query 4: Complaint Rankings Over Time (WINDOW FUNCTIONS)

How are complaint rankings shifting? This query uses `RANK()` and `SUM() OVER` window functions to track each make's complaint position year over year with a running total — identifying which manufacturers consistently dominate and whether their position is improving or worsening.

In [ ]:
%%sql -r ComplaintsRank
-- ================================================================
-- Query 4: Complaint Rankings Over Time (WINDOW FUNCTIONS)
-- ================================================================
-- Uses RANK() to determine each make's complaint position per year
-- and SUM() OVER to compute a running total of complaints.
-- Reveals which manufacturers consistently lead in complaints
-- and how their ranking shifts year over year.
-- ================================================================

SELECT
    MAKETXT AS MAKE,
    YEARTXT AS MODEL_YEAR,
    COUNT(DISTINCT CMPLID) AS COMPLAINTS,
    SUM(COUNT(DISTINCT CMPLID)) OVER (PARTITION BY MAKETXT ORDER BY YEARTXT) AS RUNNING_TOTAL_COMPLAINTS,
    RANK() OVER (PARTITION BY YEARTXT ORDER BY COUNT(DISTINCT CMPLID) DESC) AS RANK_IN_YEAR
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
GROUP BY MAKETXT, YEARTXT
ORDER BY YEARTXT, RANK_IN_YEAR;

In [ ]:
# ================================================================
# Query 4 Visualization: The Complaint Leaderboard Over Time
# ================================================================

for col in ['MODEL_YEAR', 'RANK_IN_YEAR', 'COMPLAINTS']:
    ComplaintsRank[col] = pd.to_numeric(ComplaintsRank[col], errors='coerce')
ComplaintsRank = ComplaintsRank.dropna(subset=['MODEL_YEAR', 'RANK_IN_YEAR', 'COMPLAINTS'])
ComplaintsRank = ComplaintsRank[ComplaintsRank['MODEL_YEAR'] <= 2026]

top5 = ComplaintsRank.groupby('MAKE')['COMPLAINTS'].sum().nlargest(5).index.tolist()
df_top5 = ComplaintsRank[ComplaintsRank['MAKE'].isin(top5)]
colors = dict(zip(top5, ['#e94560', '#f5a623', '#00d2d3', '#a29bfe', '#55efc4']))

plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('The Complaint Leaderboard Over Time', fontsize=14, fontweight='bold')

for make in top5:
    s = df_top5[df_top5['MAKE'] == make].sort_values('MODEL_YEAR')
    ax1.plot(s['MODEL_YEAR'], s['RANK_IN_YEAR'], label=make, color=colors[make], lw=2.5, marker='o', ms=3)
    ax2.plot(s['MODEL_YEAR'], s['COMPLAINTS'], label=make, color=colors[make], lw=2.5, marker='s', ms=3)

ax1.invert_yaxis()
ax1.set_xlabel('Model Year')
ax1.set_ylabel('Rank (1 = Most Complaints)')
ax1.set_title('Who Tops the Complaint Charts?', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.15, linestyle='--')

ax2.set_xlabel('Model Year')
ax2.set_ylabel('Complaints')
ax2.set_title('How Many Complaints Are Piling Up?', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 4 - Complaint Leaderboard Over Time.svg', format='svg', bbox_inches='tight')

### Findings: Complaint Rankings Over Time

- **Who Tops the Complaint Charts?** — Ford owns the #1 spot for most of the last decade, peaking at 7,552 complaints in model year 2020. But the leaderboard is shifting — Tesla took the crown in 2023 and Hyundai grabbed it in 2025, signaling that EV growing pains and newer manufacturers are changing the complaint landscape
- **How Many Complaints Are Piling Up?** — Volume is actually declining for legacy leaders. Ford went from 7,552 (2020) to 4,866 (2022), suggesting either quality improvements or simply fewer of those model-year vehicles still on the road. Meanwhile, newer makes are ramping up as their fleets grow

## Query 5: What Breaks and Who Makes It (GROUP BY ROLLUP)

Drilling deeper into complaints: `GROUP BY ROLLUP` produces a hierarchical view with detailed counts per make/component, subtotals per make, and a grand total. This reveals both the specific components that fail and the manufacturers responsible.

In [ ]:
%%sql -r ComponentFailurePatterns
-- ================================================================
-- Query 5: What Breaks and Who Makes It (GROUP BY ROLLUP)
-- ================================================================
-- Uses ROLLUP to create a hierarchical summary of complaints:
-- Make > Component, with automatic subtotals at each level.
-- Shows both the detail (which components fail) and the big
-- picture (total complaints per make and grand total).
-- ================================================================

SELECT
    COALESCE(MAKETXT, '-- ALL MAKES --') AS MAKE,
    COALESCE(CDESCR, '-- ALL COMPONENTS --') AS COMPONENT,
    COUNT(*) AS COMPLAINT_COUNT,
    SUM(DEATHS) AS TOTAL_DEATHS,
    SUM(INJURED) AS TOTAL_INJURIES
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
GROUP BY ROLLUP(MAKETXT, CDESCR)
ORDER BY MAKETXT NULLS LAST, COMPLAINT_COUNT DESC;

In [ ]:
# ================================================================
# Query 5 Visualization: What Breaks and Who Makes It?
# ================================================================

plt.style.use('dark_background')
ComponentFailurePatterns['COMPLAINT_COUNT'] = pd.to_numeric(ComponentFailurePatterns['COMPLAINT_COUNT'], errors='coerce')

exclude = ['-- all components --', '-- all makes --', 'unknown', 'unspecified']
df_clean = ComponentFailurePatterns[~ComponentFailurePatterns['COMPONENT'].str.lower().str.strip().isin(exclude)]
df_clean = df_clean[~df_clean['MAKE'].str.lower().str.strip().isin(exclude)]

noise_patterns = r'(?i)^no summary|^see attached|^n/a$|^none$|^na$|^not applicable$|^unknown$|^takata recall$'
df_clean = df_clean[~df_clean['COMPONENT'].str.strip().str.contains(noise_patterns, regex=True, na=False)]

consolidation_map = {
    r'(?i)^transmission.*fail': 'TRANSMISSION FAILURE',
    r'(?i)^brake.*fail': 'BRAKE FAILURE',
    r'(?i)^engine.*fail': 'ENGINE FAILURE',
    r'(?i)^power train': 'POWER TRAIN',
    r'(?i)^air bag': 'AIR BAGS',
    r'(?i)^air bags': 'AIR BAGS',
    r'(?i)^electrical.*system': 'ELECTRICAL SYSTEM',
    r'(?i)^service brake': 'SERVICE BRAKES',
    r'(?i)^steering.*': 'STEERING',
    r'(?i)^seat belt': 'SEAT BELTS',
    r'(?i)^fuel system': 'FUEL SYSTEM',
}
df_clean = df_clean.copy()
for pattern, replacement in consolidation_map.items():
    df_clean.loc[df_clean['COMPONENT'].str.strip().str.contains(pattern, regex=True, na=False), 'COMPONENT'] = replacement

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('What Breaks and Who Makes It?', fontsize=14, fontweight='bold')

make_totals = ComponentFailurePatterns[ComponentFailurePatterns['COMPONENT'] == '-- ALL COMPONENTS --'].copy()
make_totals = make_totals[make_totals['MAKE'] != '-- ALL MAKES --']
make_top10 = make_totals.nlargest(10, 'COMPLAINT_COUNT').sort_values('COMPLAINT_COUNT', ascending=True)
ax1.barh(make_top10['MAKE'], make_top10['COMPLAINT_COUNT'],
         color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, 10)), edgecolor='white', linewidth=0.3)
ax1.set_xlabel('Total Complaints', fontsize=11, fontweight='bold')
ax1.set_title('Which Brands Get the Most Complaints?', fontweight='bold')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

comp_top10 = df_clean.groupby('COMPONENT')['COMPLAINT_COUNT'].sum().nlargest(10).sort_values(ascending=True)
ax2.barh(comp_top10.index.str[:35], comp_top10.values,
         color=plt.cm.BuPu(np.linspace(0.3, 0.9, 10)), edgecolor='white', linewidth=0.3)
ax2.set_xlabel('Total Complaints', fontsize=11, fontweight='bold')
ax2.set_title('What Parts Fail the Most?', fontweight='bold')
ax2.grid(axis='x', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 5 - What Breaks and Who Makes It.svg', format='svg', bbox_inches='tight')

### Findings: Complaints Rollup Hierarchy

- **Which Brands Get the Most Complaints?** — Ford, Chevrolet, and Dodge tower over the rest. Their dominance is largely a product of fleet size — more cars on the road means more things to complain about
- **What Parts Fail the Most?** — Steering, brakes and transmissions emerge as the clear top offenders. These are high-stress mechanical systems that degrade with mileage, and their failures are immediately noticeable — and dangerous — which drives high reporting rates

---
# Part 3: Do Complaints Match Safety Ratings?
*Here's the million-dollar question: do vehicles with more complaints also score worse in crash tests? Or is complaint volume just a proxy for sales volume? A JOIN reveals the truth.*

## Query 6: Do Complaints Match Safety Ratings? (JOIN)

The key question: do vehicles with more complaints also score worse in crash tests? This query joins the complaints table with safety ratings on make/model/year to investigate whether consumer-reported issues correlate with official crash test performance.

In [ ]:
%%sql -r ComplaintsWithSafetyRatings
-- ================================================================
-- Query 6: Do Complaints Match Safety Ratings? (JOIN)
-- ================================================================
-- Joins complaints with safety ratings on make/model/year to test
-- whether vehicles with more complaints also have lower safety
-- ratings — exploring the link between consumer issues and
-- crash test performance.
-- ================================================================

SELECT
    c.MAKETXT AS MAKE,
    c.MODELTXT AS MODEL,
    c.YEARTXT AS MODEL_YEAR,
    COUNT(*) AS COMPLAINT_COUNT,
    SUM(c.DEATHS) AS TOTAL_DEATHS,
    SUM(c.INJURED) AS TOTAL_INJURIES,
    s.OVERALL_STARS,
    s.ROLLOVER_STARS
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS c
INNER JOIN BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY s
    ON UPPER(c.MAKETXT) = UPPER(s.MAKE)
    AND UPPER(c.MODELTXT) = UPPER(s.MODEL)
    AND TRY_CAST(c.YEARTXT AS NUMBER) = s.MODEL_YR
GROUP BY c.MAKETXT, c.MODELTXT, c.YEARTXT, s.OVERALL_STARS, s.ROLLOVER_STARS
ORDER BY COMPLAINT_COUNT DESC;

In [ ]:
# ================================================================
# Query 6 Visualization: Do Complaints Match Safety Ratings?
# ================================================================

plt.style.use('dark_background')
ComplaintsWithSafetyRatings['OVERALL_STARS'] = pd.to_numeric(ComplaintsWithSafetyRatings['OVERALL_STARS'], errors='coerce')
ComplaintsWithSafetyRatings['ROLLOVER_STARS'] = pd.to_numeric(ComplaintsWithSafetyRatings['ROLLOVER_STARS'], errors='coerce')
ComplaintsWithSafetyRatings['TOTAL_INJURIES'] = pd.to_numeric(ComplaintsWithSafetyRatings['TOTAL_INJURIES'], errors='coerce').fillna(0)
ComplaintsWithSafetyRatings['COMPLAINT_COUNT'] = pd.to_numeric(ComplaintsWithSafetyRatings['COMPLAINT_COUNT'], errors='coerce')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

df_top = ComplaintsWithSafetyRatings.nlargest(15, 'COMPLAINT_COUNT').sort_values('COMPLAINT_COUNT', ascending=True)
ax1.barh(range(len(df_top)), df_top['COMPLAINT_COUNT'].values,
         color=plt.cm.Reds(np.linspace(0.3, 0.9, 15)))
ax1.set_yticks(range(len(df_top)))
ax1.set_yticklabels((df_top['MAKE'] + ' ' + df_top['MODEL']).values)
ax1.set_xlabel('Complaint Count', fontsize=11, fontweight='bold')
ax1.set_title('Most Complained-About Models (with Crash Ratings)', fontweight='bold')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

df_scatter = ComplaintsWithSafetyRatings.dropna(subset=['OVERALL_STARS', 'COMPLAINT_COUNT'])
scatter = ax2.scatter(df_scatter['OVERALL_STARS'], df_scatter['COMPLAINT_COUNT'],
                      s=df_scatter['TOTAL_INJURIES'] * 2 + 20, alpha=0.6,
                      c=df_scatter['ROLLOVER_STARS'], cmap='RdYlGn', edgecolors='white', linewidths=0.5)
ax2.set_xlabel('Overall Safety Stars', fontsize=11, fontweight='bold')
ax2.set_ylabel('Complaint Count', fontsize=11, fontweight='bold')
ax2.set_title('Do Safer Cars Get Fewer Complaints?', fontweight='bold')
ax2.grid(alpha=0.15, linestyle='--')
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Rollover Stars')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 6 - Do Complaints Match Safety Ratings.svg', format='svg', bbox_inches='tight')

### Findings: Complaints vs Safety Ratings

- **Do Safer Cars Get Fewer Complaints?** — Not really. High-complaint vehicles can still have excellent safety ratings because complaint volume is driven by sales volume and component reliability, not crash performance. A 5-star car with a faulty transmission will still generate thousands of complaints
- **Where Do the Dangerous Cars Cluster?** — Bubble size (injuries) reveals the truth: the most dangerous vehicles cluster at lower star ratings in the scatter plot, confirming that NHTSA testing does identify genuinely unsafe vehicles. The rating system works — it just doesn't predict non-crash complaints
- **What About Rollover Risk?** — Color-coding by rollover stars shows some high-complaint models have poor rollover resistance despite good frontal/side scores. Consumers should check all four categories, not just the overall number

---
# Part 4: When Does NHTSA Take Notice?
*Complaints pile up, but when does the government actually act? This section examines which vehicles attract formal investigations and which recalls trigger regulatory scrutiny.*

## Query 7: Which Cars Attract Regulatory Scrutiny?

When complaints reach a critical mass, NHTSA opens formal investigations. This analysis identifies the specific make/model/component combinations that have attracted the most investigative attention — revealing persistent defects that trigger repeated regulatory reviews.

In [ ]:
%%sql -r RankingMakesByInvestigations
-- ================================================================
-- Query 7: Which Cars Attract Regulatory Scrutiny?
-- ================================================================
-- Groups investigations by make/model/component to find which
-- specific vehicle configurations attract the most formal NHTSA
-- scrutiny. Related campaign counts show whether investigations
-- eventually led to recalls.
-- ================================================================

SELECT
    MAKE,
    MODEL,
    COMPNAME AS COMPONENT,
    COUNT(DISTINCT ACTION_NUMBER) AS INVESTIGATION_COUNT,
    COUNT(DISTINCT CAMPNO) AS RELATED_CAMPAIGNS,
    MIN(YEAR) AS EARLIEST_YEAR,
    MAX(YEAR) AS LATEST_YEAR
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_INVESTIGATIONS
GROUP BY MAKE, MODEL, COMPNAME
ORDER BY INVESTIGATION_COUNT DESC
LIMIT 20;

In [ ]:
# ================================================================
# Query 7 Visualization: Which Cars Attract Regulatory Scrutiny?
# ================================================================

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 7))

df_inv = RankingMakesByInvestigations.dropna(subset=['MAKE', 'MODEL', 'COMPONENT']).head(15)
labels = df_inv['MAKE'].astype(str) + ' ' + df_inv['MODEL'].astype(str) + ' (' + df_inv['COMPONENT'].astype(str) + ')'

bars = ax.barh(labels[::-1], df_inv['INVESTIGATION_COUNT'][::-1],
               color=plt.cm.Purples(np.linspace(0.4, 0.9, 15)),
               edgecolor='white', linewidth=0.3)

ax.set_xlabel('Number of Investigations', fontsize=12, fontweight='bold')
ax.set_title('Which Cars Attract the Most Regulatory Scrutiny?', fontsize=16, fontweight='bold', pad=15)
ax.bar_label(bars, padding=3, fontsize=9)
ax.grid(axis='x', alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 7 - Which Cars Attract Scrutiny.svg', format='svg', bbox_inches='tight')

### Findings: Ranking Makes by Investigations

- **Which Combinations Attract the Most Scrutiny?** — Repeated investigations into the same make/model/component suggest persistent or unresolved defects that generate enough incident reports to trigger multiple formal reviews by NHTSA
- **Do Investigations Lead to Recalls?** — The related campaign count column reveals the answer: high investigation counts with zero related campaigns indicate regulatory scrutiny that didn't result in formal action, while high counts with campaigns show the pipeline working as intended

---
# Part 5: What Gets Recalled?
*Now we look at the outcome: which brands recall the most vehicles, what components drive those recalls, and how does the investigation pipeline connect to recall action?*

## Query 8: Who Puts the Most Cars at Risk? (Recalls Ranking)

Moving from investigations to action: which manufacturers recall the most vehicles? Using campaign-level deduplication (QUALIFY + ROW_NUMBER), we isolate which brands put the most vehicles at risk through confirmed safety defects.

In [ ]:
%%sql -r RankingMakesByRecalls
-- ================================================================
-- Query 8: Who Puts the Most Cars at Risk? (Recalls Ranking)
-- ================================================================
-- Deduplicates recall campaigns at the make level using QUALIFY
-- and ROW_NUMBER to prevent double-counting vehicles affected.
-- Produces the top 20 makes by total vehicles recalled, with
-- severity flags for catastrophic defects.
-- ================================================================

WITH campaign_level AS (
    SELECT
        CAMPNO,
        MAKETXT,
        POTAFF,
        COMPNAME,
        DO_NOT_DRIVE,
        PARK_OUTSIDE,
        ROW_NUMBER() OVER (PARTITION BY CAMPNO ORDER BY MAKETXT) AS rn
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY CAMPNO, MAKETXT ORDER BY MODELTXT) = 1
)
SELECT
    MAKETXT AS MAKE,
    COUNT(DISTINCT CAMPNO) AS TOTAL_RECALLS,
    SUM(CASE WHEN rn = 1 THEN POTAFF ELSE 0 END) AS TOTAL_VEHICLES_AFFECTED,
    COUNT(DISTINCT COMPNAME) AS COMPONENTS_RECALLED,
    SUM(CASE WHEN DO_NOT_DRIVE = 'Y' THEN 1 ELSE 0 END) AS DO_NOT_DRIVE_RECALLS,
    SUM(CASE WHEN PARK_OUTSIDE = 'Y' THEN 1 ELSE 0 END) AS PARK_OUTSIDE_RECALLS
FROM campaign_level
GROUP BY MAKETXT
ORDER BY TOTAL_VEHICLES_AFFECTED DESC NULLS LAST
LIMIT 20;

In [ ]:
# ================================================================
# Query 8 Visualization: Who Puts the Most Cars at Risk?
# ================================================================

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(16, 7))

df_top_recalls = RankingMakesByRecalls.head(15)
bars = ax.barh(
    df_top_recalls['MAKE'][::-1],
    df_top_recalls['TOTAL_VEHICLES_AFFECTED'][::-1],
    color='#00d2d3', edgecolor='white', linewidth=0.3
)

ax.set_xlabel('Total Vehicles Affected', fontsize=12, fontweight='bold')
ax.set_title('Who Puts the Most Cars at Risk?', fontsize=16, fontweight='bold', pad=15)
ax.bar_label(bars, fmt='{:,.0f}', padding=5, fontsize=8)
ax.set_xlim(0, df_top_recalls['TOTAL_VEHICLES_AFFECTED'].max() * 1.15)
ax.grid(axis='x', alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 8 - Who Puts the Most Cars at Risk.svg', format='svg', bbox_inches='tight')

### Findings: Ranking Makes by Recalls

- **Who Recalls the Most Vehicles?** — The same brands topping the complaint charts also lead in recalls — Ford, Toyota, Chevrolet. This confirms that market share is the primary driver of both metrics
- **What Are the Severity Flags?** — The "Do Not Drive" and "Park Outside" columns separate routine software fixes from life-threatening defects. Brands with high severity flags relative to their total recall volume have engineering problems that go beyond normal wear and tear

## Query 9: Recall Anatomy by Component × Make (GROUP BY CUBE)

`GROUP BY CUBE` produces every possible combination of component and make groupings — component subtotals, make subtotals, cross-tabulations, and a grand total. This reveals which components trigger the biggest recalls and which manufacturers recall the most cars.

In [ ]:
%%sql -r RecallCubeAnalysis
-- ================================================================
-- Query 9: Recall Anatomy by Component × Make (GROUP BY CUBE)
-- ================================================================
-- Uses CUBE to generate all possible grouping combinations of
-- component and make. Produces: detail rows (component+make),
-- component subtotals (across all makes), make subtotals (across
-- all components), and the grand total.
-- ================================================================

SELECT
    COALESCE(COMPNAME, '-- ALL COMPONENTS --') AS COMPONENT,
    COALESCE(MAKETXT, '-- ALL MAKES --') AS MAKE,
    COUNT(DISTINCT CAMPNO) AS RECALL_COUNT,
    SUM(POTAFF) AS TOTAL_AFFECTED
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL
WHERE MAKETXT IN ('FORD', 'CHEVROLET', 'TOYOTA', 'HONDA', 'NISSAN')
GROUP BY CUBE(COMPNAME, MAKETXT)
ORDER BY TOTAL_AFFECTED DESC NULLS LAST;

In [ ]:
# ================================================================
# Query 9 Visualization: The Recall Anatomy
# ================================================================

plt.style.use('dark_background')
RecallCubeAnalysis['RECALL_COUNT'] = pd.to_numeric(RecallCubeAnalysis['RECALL_COUNT'], errors='coerce')
RecallCubeAnalysis['TOTAL_AFFECTED'] = pd.to_numeric(RecallCubeAnalysis['TOTAL_AFFECTED'], errors='coerce')

df_detail = RecallCubeAnalysis[(RecallCubeAnalysis['MAKE'] != '-- ALL MAKES --') &
                               (RecallCubeAnalysis['COMPONENT'] != '-- ALL COMPONENTS --')].copy()

noise_patterns = r'(?i)LABELS|ADAPTIVE/MOBILITY|EQUIPMENT:OTHER|UNKNOWN|OTHER/I AM NOT SURE'
df_detail = df_detail[~df_detail['COMPONENT'].str.contains(noise_patterns, regex=True, na=False)]

df_detail['COMPONENT_GROUP'] = df_detail['COMPONENT'].apply(lambda x: x.split(':')[0].strip())

comp_agg = df_detail.groupby('COMPONENT_GROUP')['RECALL_COUNT'].sum().nlargest(10).sort_values(ascending=True)
make_agg = df_detail.groupby('MAKE')['RECALL_COUNT'].sum().sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('The Recall Anatomy: Components vs Manufacturers', fontsize=14, fontweight='bold')

ax1.barh(comp_agg.index.str[:30], comp_agg.values,
         color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, 10)), edgecolor='white', linewidth=0.3)
ax1.set_xlabel('Recall Campaigns', fontsize=11, fontweight='bold')
ax1.set_title('Which Components Trigger the Most Recalls?', fontweight='bold')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

ax2.barh(make_agg.index, make_agg.values,
         color=plt.cm.Set2(np.linspace(0, 1, len(make_agg))), edgecolor='white', linewidth=0.3)
ax2.set_xlabel('Recall Campaigns', fontsize=11, fontweight='bold')
ax2.set_title('Which Manufacturer Issues the Most Recalls?', fontweight='bold')
ax2.grid(axis='x', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 9 - The Recall Anatomy.svg', format='svg', bbox_inches='tight')

### Findings: Recall CUBE Analysis

- **Which Components Trigger the Most Recalls?** — After consolidating sub-variants, fuel systems (387 campaigns), electrical systems (357), hydraulic brakes (357), air bags (329), and power train (328) lead. The top five are tightly clustered — no single system dominates, suggesting recalls are a broad-based reliability problem rather than one catastrophic weak link
- **Which Manufacturer Issues the Most Recalls?** — Ford leads in total campaign count, reflecting both its massive fleet size and a broader range of affected components. Honda and Toyota follow, while Nissan has the fewest campaigns among the five

## Query 10: Recalls Linked to Investigations (JOIN)

Closing the loop: this query joins recall records with formal NHTSA investigation data using campaign numbers. It reveals which recalls escalated into full investigations, showing the severity threshold that triggers regulatory action.

In [ ]:
%%sql -r RecallsLinkedToInvestigations
-- ================================================================
-- Query 10: Recalls Linked to Investigations (JOIN)
-- ================================================================
-- Joins recalls with NHTSA investigations on campaign number to
-- identify which recalls triggered formal investigations. Shows
-- the relationship between recall severity and regulatory scrutiny.
-- ================================================================

SELECT
    r.MAKETXT AS MAKE,
    r.CAMPNO AS CAMPAIGN_NUMBER,
    r.COMPNAME AS RECALL_COMPONENT,
    r.POTAFF AS VEHICLES_AFFECTED,
    r.DESC_DEFECT,
    i.ACTION_NUMBER,
    i.SUBJECT AS INVESTIGATION_SUBJECT,
    i.SUMMARY AS INVESTIGATION_SUMMARY
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL r
INNER JOIN BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_INVESTIGATIONS i
    ON r.CAMPNO = i.CAMPNO
ORDER BY r.POTAFF DESC;

In [ ]:
# ================================================================
# Query 10 Visualization: Recalls Linked to Investigations
# ================================================================

plt.style.use('dark_background')
RecallsLinkedToInvestigations['VEHICLES_AFFECTED'] = pd.to_numeric(RecallsLinkedToInvestigations['VEHICLES_AFFECTED'], errors='coerce')

df_recalls_dedup = RecallsLinkedToInvestigations.drop_duplicates(subset=['CAMPAIGN_NUMBER'])
df_inv_dedup = RecallsLinkedToInvestigations.drop_duplicates(subset=['ACTION_NUMBER'])

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('From Complaint to Investigation: The Recall Pipeline', fontsize=14, fontweight='bold')

inv_by_make = df_inv_dedup.groupby('MAKE')['ACTION_NUMBER'].nunique().nlargest(10).sort_values()
axes[0,0].barh(inv_by_make.index, inv_by_make.values, color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, 10)))
axes[0,0].set_xlabel('Unique Investigations')
axes[0,0].set_title('Who Gets Investigated the Most?', fontweight='bold')
axes[0,0].grid(axis='x', alpha=0.15, linestyle='--')

aff_by_comp = df_recalls_dedup.groupby('RECALL_COMPONENT')['VEHICLES_AFFECTED'].sum().nlargest(10).sort_values()
axes[0,1].barh([c[:28] for c in aff_by_comp.index], aff_by_comp.values, color=plt.cm.BuPu(np.linspace(0.3, 0.9, 10)))
axes[0,1].set_xlabel('Vehicles Affected')
axes[0,1].set_title('Which Parts Put the Most Cars at Risk?', fontweight='bold')
axes[0,1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
axes[0,1].grid(axis='x', alpha=0.15, linestyle='--')

top_makes = df_recalls_dedup.groupby('MAKE')['CAMPAIGN_NUMBER'].nunique().nlargest(10)
inv_per_make = df_inv_dedup.groupby('MAKE')['ACTION_NUMBER'].nunique()
x_pos = np.arange(len(top_makes))
axes[1,0].bar(x_pos - 0.2, top_makes.values, 0.4, color='#e94560', label='Recalls')
axes[1,0].bar(x_pos + 0.2, [inv_per_make.get(m, 0) for m in top_makes.index], 0.4, color='#00d2d3', label='Investigations')
axes[1,0].set_xticks(x_pos)
axes[1,0].set_xticklabels(top_makes.index, rotation=45, ha='right', fontsize=8)
axes[1,0].set_title('Do More Recalls Mean More Investigations?', fontweight='bold')
axes[1,0].legend(fontsize=8)
axes[1,0].grid(axis='y', alpha=0.15, linestyle='--')

aff_vals = df_recalls_dedup['VEHICLES_AFFECTED'].dropna()
aff_vals = aff_vals[aff_vals > 0]
axes[1,1].hist(np.log10(aff_vals), bins=30, color='#a29bfe', edgecolor='white', linewidth=0.3)
axes[1,1].axvline(np.log10(aff_vals.median()), color='#ffd700', linestyle='--', lw=2, label=f'Median: {int(aff_vals.median()):,}')
axes[1,1].set_xlabel('Vehicles Affected (log10)')
axes[1,1].set_ylabel('# Recalls (in 1000s)')
axes[1,1].set_title('How Big Is a Typical Recall?', fontweight='bold')
axes[1,1].legend(fontsize=9)
axes[1,1].grid(axis='y', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 10 - Recalls Linked to Investigations.svg', format='svg', bbox_inches='tight')

### Findings: Recalls Linked to Investigations

- **Who Gets Investigated the Most?** — Ford, Chevrolet, and Dodge dominate the investigation count. These are also the highest-volume manufacturers, so more recalls naturally means more regulatory touchpoints — but their consistent presence suggests systemic supply-chain issues rather than bad luck
- **Which Parts Put the Most Cars at Risk?** — Tires (tread/belt) lead at ~ 18M vehicles affected, followed by vehicle speed control (~ 15M) and front seat belt buckle assemblies (~ 14.5M). These are supplier-level defects shared across multiple brands — a single faulty component design can ripple across millions of vehicles simultaneously
- **Do More Recalls Mean More Investigations?** — Largely yes — the grouped bar shows a strong correlation between recall count and investigation count. But some brands show a higher investigation-to-recall ratio, suggesting their defects attract more regulatory scrutiny per event
- **How Big Is a Typical Recall?** — The log-scale histogram reveals that most recalls affect 10K–100K vehicles, but rare mega-recalls in the millions create a long right tail. The dashed median line is far more representative than any mean figure

---
# Part 6: The Full Pipeline — From Complaint to Recall
*Bringing all four NHTSA tables together with the most advanced SQL — EXISTS subqueries, dynamic CTEs, CASE WHEN risk classification, 3D CUBE, and correlated subqueries — to trace the complete lifecycle of a safety defect.*

## Query 11: The Full Pipeline — Do Complaints Lead to Recalls? (JOIN)

The complete pipeline test: joining complaints to recalls on make/model/year reveals which manufacturers' complaints actually led to formal recall campaigns — and which brands may have unaddressed safety concerns still in the field.

In [ ]:
%%sql -r ComplaintsLinkedToRecalls
-- ================================================================
-- Query 11: The Full Pipeline — Complaints to Recalls (JOIN)
-- ================================================================
-- LEFT JOINs complaints to recalls on make/model/year to trace
-- which consumer-reported issues eventually led to formal recall
-- campaigns. Aggregates by make to show complaint volume, related
-- recalls, and severity metrics.
-- ================================================================

SELECT
    c.MAKETXT AS MAKE,
    COUNT(DISTINCT c.CMPLID) AS COMPLAINT_COUNT,
    COUNT(DISTINCT r.CAMPNO) AS RELATED_RECALLS,
    SUM(c.DEATHS) AS DEATHS_FROM_COMPLAINTS,
    SUM(c.INJURED) AS INJURIES_FROM_COMPLAINTS
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS c
LEFT JOIN BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL r
    ON UPPER(c.MAKETXT) = UPPER(r.MAKETXT)
    AND UPPER(c.MODELTXT) = UPPER(r.MODELTXT)
    AND c.YEARTXT = r.YEARTXT
WHERE c.MAKETXT IS NOT NULL
    AND UPPER(TRIM(c.MAKETXT)) NOT IN ('UNKNOWN', 'OTHER', '')
GROUP BY c.MAKETXT
HAVING COUNT(DISTINCT c.CMPLID) > 100
ORDER BY COMPLAINT_COUNT DESC
LIMIT 20;

In [ ]:
# ================================================================
# Query 11 Visualization: Do Complaints Lead to Recalls?
# ================================================================

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(16, 8))

ComplaintsLinkedToRecalls['COMPLAINT_COUNT'] = pd.to_numeric(ComplaintsLinkedToRecalls['COMPLAINT_COUNT'], errors='coerce')
ComplaintsLinkedToRecalls['RELATED_RECALLS'] = pd.to_numeric(ComplaintsLinkedToRecalls['RELATED_RECALLS'], errors='coerce')

df_plot = ComplaintsLinkedToRecalls.head(15).sort_values('COMPLAINT_COUNT', ascending=True)

bars = ax.barh(
    df_plot['MAKE'],
    df_plot['COMPLAINT_COUNT'],
    color=plt.cm.OrRd(np.linspace(0.3, 0.9, 15)),
    edgecolor='white', linewidth=0.3
)

ax.set_xlabel('Total Complaints', fontsize=12, fontweight='bold')
ax.set_title('Do Complaints Actually Lead to Recalls?', fontsize=16, fontweight='bold', pad=15)
ax.bar_label(bars, fmt='{:,.0f}', padding=-45, fontsize=9, color='white', fontweight='bold')

for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.text(
        row['COMPLAINT_COUNT'] + df_plot['COMPLAINT_COUNT'].max() * 0.01,
        i,
        f"{int(row['RELATED_RECALLS']):,} recalls",
        va='center', fontsize=9, color='#00d2d3', fontweight='bold'
    )

ax.set_xlim(0, df_plot['COMPLAINT_COUNT'].max() * 1.18)
ax.grid(axis='x', alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 11 - Do Complaints Lead to Recalls.svg', format='svg', bbox_inches='tight')

### Findings: Linking Complaints to Recalls

- **Which Brands Get Recalled After Complaints?** — Ford leads in both raw complaints and related recall campaigns, confirming the pipeline from consumer report to manufacturer action is active. The annotation shows how many unique recall campaigns each brand's complaints ultimately triggered
- **What's the Complaint-to-Recall Ratio?** — A brand with high complaints but few recalls may be ignoring consumer reports. Conversely, brands with many recalls relative to complaints are being proactive (or facing regulatory pressure) to address issues before they escalate

## Query 12: Strategic Safety Matrix — Crash Rate vs Casualty Rate (ROLLUP)

Generates a hierarchical view of the Top 5 makes with crash and casualty rates per component. The bubble scatter reveals which brand/component pairs are both frequent AND deadly.

In [ ]:
%%sql -r ComplaintsHierarchyRollup
-- ================================================================
-- Query 12: Complaints Hierarchy (ROLLUP)
-- ================================================================
-- Dynamically identifies the Top 5 complained-about makes, then
-- applies CASE WHEN component normalization and ROLLUP to produce
-- make subtotals, component subtotals, and a grand total. Crash
-- rate and casualty rate reveal true danger beyond raw volume.
-- ================================================================

WITH Dynamic_Top_Makes AS (
    SELECT MAKETXT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
    WHERE MAKETXT IS NOT NULL AND PROD_TYPE = 'V'
    GROUP BY MAKETXT
    ORDER BY COUNT(*) DESC
    LIMIT 5
),
Filtered_Complaints AS (
    SELECT 
        UPPER(TRIM(MAKETXT)) AS CLEAN_MAKE,
        DEATHS,
        INJURED,
        CRASH,
        CASE 
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('SERVICE BRAKES, HYDRAULIC', 'SERVICE BRAKES', 'SERVICE BRAKES, AIR', 'SERVICE BRAKES, ELECTRIC', 'PARKING BRAKE') THEN 'BRAKES & BRAKING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('ENGINE', 'ENGINE AND ENGINE COOLING') THEN 'ENGINE & COOLING'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('FUEL SYSTEM, GASOLINE', 'FUEL/PROPULSION SYSTEM', 'FUEL SYSTEM, OTHER', 'FUEL SYSTEM, DIESEL', 'HYBRID PROPULSION SYSTEM') THEN 'FUEL & PROPULSION SYSTEM'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('ELECTRONIC STABILITY CONTROL (ESC)', 'ELECTRONIC STABILITY CONTROL', 'TRACTION CONTROL SYSTEM') THEN 'ELECTRONIC STABILITY & TRACTION'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('VISIBILITY', 'VISIBILITY/WIPER') THEN 'VISIBILITY & WIPERS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('EXTERIOR LIGHTING', 'INTERIOR LIGHTING') OR UPPER(TRIM(COMPDESC)) LIKE '%LIGHTING%' THEN 'LIGHTING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('TIRES', 'WHEELS') THEN 'TIRES & WHEELS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('FORWARD COLLISION AVOIDANCE', 'LANE DEPARTURE', 'BACK OVER PREVENTION', 'COMMUNICATION') THEN 'ADAS & ADVANCED TECHNOLOGY'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('EQUIPMENT', 'EQUIPMENT ADAPTIVE/MOBILITY', 'TRAILER HITCHES') THEN 'EQUIPMENT & ACCESSORIES'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('CHILD SEAT', 'CHEST CLIP, BUCKLE, HARNESS', 'CARRY HANDLE, SHELL, BASE', 'TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)', 'INSERT, PADDING') THEN 'CHILD SEAT & RESTRAINT EQUIPMENT'
            WHEN UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) IN ('UNKNOWN OR OTHER', 'OTHER/I AM NOT SURE', 'OTHER/UNKNOWN', 'NONE', '1', '120', 'FIRERELATED', '') OR SPLIT_PART(COMPDESC, ':', 1) IS NULL THEN 'OTHER / UNKNOWN'
            ELSE UPPER(TRIM(SPLIT_PART(COMPDESC, ':', 1))) 
        END AS CLEAN_COMPONENT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
    WHERE PROD_TYPE = 'V'
      AND UPPER(TRIM(MAKETXT)) IN (SELECT MAKETXT FROM Dynamic_Top_Makes)
)
SELECT
    CASE WHEN GROUPING(CLEAN_MAKE) = 1 THEN '--- ALL MAKES (GRAND TOTAL) ---' ELSE CLEAN_MAKE END AS VEHICLE_MAKE,
    CASE 
        WHEN GROUPING(CLEAN_MAKE) = 1 AND GROUPING(CLEAN_COMPONENT) = 1 THEN '--- ALL COMPONENTS (GRAND TOTAL) ---'
        WHEN GROUPING(CLEAN_COMPONENT) = 1 THEN '--- ALL COMPONENTS (SUBTOTAL) ---' 
        ELSE CLEAN_COMPONENT 
    END AS PRIMARY_COMPONENT,
    COUNT(*) AS TOTAL_COMPLAINTS,
    SUM(DEATHS) AS TOTAL_DEATHS,
    SUM(INJURED) AS TOTAL_INJURIES,
    COUNT_IF(CRASH = 'Y') AS TOTAL_CRASHES,
    ROUND(COUNT_IF(CRASH = 'Y') * 100.0 / NULLIF(COUNT(*), 0), 2) AS CRASH_RATE_PCT,
    ROUND((SUM(DEATHS) + SUM(INJURED)) * 100.0 / NULLIF(COUNT(*), 0), 2) AS CASUALTY_RATE_PCT
FROM Filtered_Complaints
GROUP BY ROLLUP(CLEAN_MAKE, CLEAN_COMPONENT)
ORDER BY GROUPING(CLEAN_MAKE) ASC, VEHICLE_MAKE ASC, GROUPING(CLEAN_COMPONENT) ASC, TOTAL_COMPLAINTS DESC;

In [ ]:
# ================================================================
# Query 12 Visualization: Strategic Safety Matrix (Scatter)
# ================================================================

plt.style.use('dark_background')
df_clean = ComplaintsHierarchyRollup.copy()
df_clean = df_clean[(df_clean['PRIMARY_COMPONENT'] != 'OTHER / UNKNOWN') & 
                    (df_clean['PRIMARY_COMPONENT'] != '--- ALL COMPONENTS (SUBTOTAL) ---') &
                    (df_clean['PRIMARY_COMPONENT'] != '--- ALL COMPONENTS (GRAND TOTAL) ---')].copy()
df_clean['CRASH_RATE_PCT'] = pd.to_numeric(df_clean['CRASH_RATE_PCT'], errors='coerce')
df_clean['CASUALTY_RATE_PCT'] = pd.to_numeric(df_clean['CASUALTY_RATE_PCT'], errors='coerce')
df_clean['TOTAL_COMPLAINTS'] = pd.to_numeric(df_clean['TOTAL_COMPLAINTS'], errors='coerce')

fig, ax = plt.subplots(figsize=(14, 9))

scatter = ax.scatter(
    x=df_clean['CRASH_RATE_PCT'],
    y=df_clean['CASUALTY_RATE_PCT'],
    s=df_clean['TOTAL_COMPLAINTS'] * 0.04,
    c=df_clean['VEHICLE_MAKE'].map({
        'FORD': '#1f77b4', 'CHEVROLET': '#ff7f0e',
        'TOYOTA': '#2ca02c', 'DODGE': '#d62728', 'JEEP': '#9467bd'
    }),
    alpha=0.65, edgecolors='w', linewidth=1.2
)

ax.set_title('Which Brand/Component Combos Are Both Common AND Deadly?', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Real-World Crash Rate (%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Casualty Rate (%) [Deaths + Injuries]', fontsize=12, fontweight='bold')

handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10)
           for c in ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']]
ax.legend(handles, ['FORD', 'CHEVROLET', 'TOYOTA', 'DODGE', 'JEEP'], title='Vehicle Manufacturer', loc='upper left')
ax.text(0.98, 0.02, '*Bubble Size = Complaint Volume',
        transform=ax.transAxes, fontsize=9, style='italic', ha='right')
ax.grid(alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 12 - Common AND Deadly Combinations.svg', format='svg', bbox_inches='tight')

### Findings: Complaints Hierarchy (ROLLUP)

- **Which Brand/Component Pairs Are Deadliest?** — Ford Tires & Wheels leads with 17,514 complaints, 327 deaths, and 2,019 injuries — the single most lethal pairing in the entire database. This isn't just a volume story; the death rate per complaint is abnormally high for this specific combination
- **What's the Crash Rate Champion?** — Toyota Vehicle Speed Control hits a 32% Crash Rate, meaning nearly 1 in 3 complaints about this system resulted in an actual collision. That's terrifying for a component designed to control how fast the car goes
- **Do Airbags Save or Kill?** — Airbags across all 5 makes maintain crash/casualty rates above 30%. This seems counterintuitive until you realize: airbag complaints are filed *after* a crash has already happened. The correlation is with crash severity, not airbag failure

## Query 13: 3-Dimensional CUBE (Brand × Component × Recall Type)

Performs a 3-dimensional cross-tabulation of recall campaigns across Brand, Component, and Recall Classification simultaneously. Uses campaign-level MAX(POTAFF) deduplication to prevent vehicle multi-counting.

In [ ]:
%%sql -r CubeAnalysis3D
-- ================================================================
-- Query 13: 3-Dimensional CUBE (Brand × Component × Recall Type)
-- ================================================================
-- Performs a 3-dimensional cross-tabulation using CUBE across
-- Brand, Component, and Recall Type simultaneously. Pre-aggregates
-- campaigns using MAX(POTAFF) to prevent vehicle double-counting,
-- then applies GROUPING() labels for readable subtotal rows.
-- ================================================================

WITH Dynamic_Top_Makes AS (
    SELECT MAKETXT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
    WHERE MAKETXT IS NOT NULL AND PROD_TYPE = 'V'
    GROUP BY MAKETXT
    ORDER BY COUNT(*) DESC
    LIMIT 5
),
Normalized_Recalls_Base AS (
    SELECT 
        UPPER(TRIM(MAKETXT)) AS CLEAN_MAKE,
        UPPER(TRIM(RCLTYPECD)) AS CLEAN_RECALL_TYPE,
        UPPER(TRIM(CAMPNO)) AS CLEAN_CAMPNO,
        POTAFF,
        CASE 
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('SERVICE BRAKES, HYDRAULIC', 'SERVICE BRAKES', 'SERVICE BRAKES, AIR', 'SERVICE BRAKES, ELECTRIC', 'PARKING BRAKE') THEN 'BRAKES & BRAKING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('ENGINE', 'ENGINE AND ENGINE COOLING') THEN 'ENGINE & COOLING'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('FUEL SYSTEM, GASOLINE', 'FUEL/PROPULSION SYSTEM', 'FUEL SYSTEM, OTHER', 'FUEL SYSTEM, DIESEL', 'HYBRID PROPULSION SYSTEM') THEN 'FUEL & PROPULSION SYSTEM'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('ELECTRONIC STABILITY CONTROL (ESC)', 'ELECTRONIC STABILITY CONTROL', 'TRACTION CONTROL SYSTEM') THEN 'ELECTRONIC STABILITY & TRACTION'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('VISIBILITY', 'VISIBILITY/WIPER') THEN 'VISIBILITY & WIPERS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('EXTERIOR LIGHTING', 'INTERIOR LIGHTING') THEN 'LIGHTING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('TIRES', 'WHEELS') THEN 'TIRES & WHEELS'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('FORWARD COLLISION AVOIDANCE', 'LANE DEPARTURE', 'BACK OVER PREVENTION', 'COMMUNICATION') THEN 'ADAS & ADVANCED TECHNOLOGY'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('EQUIPMENT', 'EQUIPMENT ADAPTIVE/MOBILITY', 'TRAILER HITCHES') THEN 'EQUIPMENT & ACCESSORIES'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('CHILD SEAT', 'CHEST CLIP, BUCKLE, HARNESS', 'CARRY HANDLE, SHELL, BASE', 'TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)', 'INSERT, PADDING') THEN 'CHILD SEAT & RESTRAINT EQUIPMENT'
            WHEN UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) IN ('UNKNOWN OR OTHER', 'OTHER/I AM NOT SURE', 'OTHER/UNKNOWN', 'NONE', '1', '120', 'FIRERELATED', '') OR COMPNAME IS NULL THEN 'OTHER / UNKNOWN'
            ELSE UPPER(TRIM(SPLIT_PART(COMPNAME, ':', 1))) 
        END AS CLEAN_COMPONENT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL
    WHERE UPPER(TRIM(MAKETXT)) IN (SELECT MAKETXT FROM Dynamic_Top_Makes)
      AND RCLTYPECD IS NOT NULL AND RCLTYPECD != 'X'
),
PreAggregated_Recalls AS (
    SELECT 
        CLEAN_MAKE, CLEAN_COMPONENT, CLEAN_RECALL_TYPE, CLEAN_CAMPNO,
        MAX(POTAFF) AS CAMPAIGN_POTAFF
    FROM Normalized_Recalls_Base
    WHERE CLEAN_COMPONENT IS NOT NULL AND CLEAN_COMPONENT != ''
    GROUP BY 1, 2, 3, 4
)
SELECT
    CASE WHEN GROUPING(CLEAN_MAKE) = 1 THEN '--- ALL MAKES ---' ELSE CLEAN_MAKE END AS VEHICLE_MAKE,
    CASE WHEN GROUPING(CLEAN_COMPONENT) = 1 THEN '--- ALL COMPONENTS ---' ELSE CLEAN_COMPONENT END AS PRIMARY_COMPONENT,
    CASE 
        WHEN GROUPING(CLEAN_RECALL_TYPE) = 1 THEN '--- ALL RECALL TYPES ---'
        WHEN CLEAN_RECALL_TYPE = 'V' THEN 'VEHICLE RECALL'
        WHEN CLEAN_RECALL_TYPE = 'E' THEN 'EQUIPMENT RECALL'
        WHEN CLEAN_RECALL_TYPE = 'T' THEN 'TIRE RECALL'
        WHEN CLEAN_RECALL_TYPE = 'C' THEN 'CHILD SEAT RECALL'
        ELSE CLEAN_RECALL_TYPE 
    END AS RECALL_CLASSIFICATION,
    COUNT(DISTINCT CLEAN_CAMPNO) AS TOTAL_RECALL_CAMPAIGNS,
    SUM(CAMPAIGN_POTAFF) AS TOTAL_VEHICLES_AFFECTED,
    ROUND(SUM(CAMPAIGN_POTAFF) * 1.0 / NULLIF(COUNT(DISTINCT CLEAN_CAMPNO), 0), 0) AS AVG_VEHICLES_PER_RECALL
FROM PreAggregated_Recalls
GROUP BY CUBE(CLEAN_MAKE, CLEAN_COMPONENT, CLEAN_RECALL_TYPE)
ORDER BY GROUPING(CLEAN_MAKE) ASC, GROUPING(CLEAN_COMPONENT) ASC, GROUPING(CLEAN_RECALL_TYPE) ASC, TOTAL_VEHICLES_AFFECTED DESC NULLS LAST
LIMIT 40;

In [ ]:
# ================================================================
# Query 13 Visualization: CUBE Matrix — Campaign vs Affected Scale
# ================================================================

plt.style.use('dark_background')
CubeAnalysis3D['TOTAL_RECALL_CAMPAIGNS'] = pd.to_numeric(CubeAnalysis3D['TOTAL_RECALL_CAMPAIGNS'], errors='coerce')
CubeAnalysis3D['TOTAL_VEHICLES_AFFECTED'] = pd.to_numeric(CubeAnalysis3D['TOTAL_VEHICLES_AFFECTED'], errors='coerce')
CubeAnalysis3D['AVG_VEHICLES_PER_RECALL'] = pd.to_numeric(CubeAnalysis3D['AVG_VEHICLES_PER_RECALL'], errors='coerce')

df_detail = CubeAnalysis3D[(CubeAnalysis3D['VEHICLE_MAKE'] != '--- ALL MAKES ---') & 
                    (CubeAnalysis3D['PRIMARY_COMPONENT'] != '--- ALL COMPONENTS ---') &
                    (CubeAnalysis3D['RECALL_CLASSIFICATION'] != '--- ALL RECALL TYPES ---')]

brand_palette = {'FORD': '#1f77b4', 'CHEVROLET': '#ff7f0e', 'TOYOTA': '#2ca02c', 'DODGE': '#d62728', 'JEEP': '#9467bd'}

fig, ax = plt.subplots(figsize=(14, 9))

scatter = ax.scatter(
    x=df_detail['TOTAL_RECALL_CAMPAIGNS'],
    y=df_detail['TOTAL_VEHICLES_AFFECTED'] / 1e6,
    s=df_detail['AVG_VEHICLES_PER_RECALL'] * 0.002,
    c=df_detail['VEHICLE_MAKE'].map(brand_palette),
    alpha=0.7, edgecolors='w', linewidth=1.2
)

ax.set_title('How Many Campaigns Does It Take to Fix a Problem?', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Total Recall Campaigns Conducted', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Vehicles Affected (Millions)', fontsize=12, fontweight='bold')

handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=11) for c in brand_palette.values()]
ax.legend(handles, brand_palette.keys(), title='Vehicle Manufacturer', loc='upper right')
ax.text(0.98, 0.02, '*Bubble Area = Average Vehicles Per Recall Campaign',
        transform=ax.transAxes, fontsize=9, style='italic', ha='right')
ax.grid(alpha=0.15, linestyle='--')
plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 13 - How Many Campaigns to Fix a Problem.svg', format='svg', bbox_inches='tight')

### Findings: 3-Dimensional CUBE Analysis

- **Who Has the Biggest Blast Radius Per Recall?** — Ford Tires & Wheels averages ~ 4.9M vehicles per recall campaign (driven by the historic Firestone/Explorer disaster). Toyota Vehicle Speed Control is even more concentrated: only 11 campaigns but affecting 10.7M vehicles (~ 970K per recall)
- **What Does the CUBE Reveal That ROLLUP Can't?** — By crossing all three dimensions simultaneously, we see that Vehicle Recalls ('V' type) account for 95%+ of affected vehicles. Equipment and tire recalls barely register in comparison — the regulatory focus is overwhelmingly on the vehicles themselves

## Query 14: Above-Average Complaint Benchmark (HAVING + Subquery)

Identifies brands whose complaint footprint exceeds the national average using a dynamic HAVING subquery. Reveals crash rates and casualty rates that separate true engineering danger from mere volume effects.

In [ ]:
%%sql -r AboveAvgComplaintBenchmark
-- ================================================================
-- Query 14: Above-Average Consumer Complaint Benchmark
-- ================================================================
-- Uses HAVING with a dynamic subquery to identify only brands
-- whose total complaint count exceeds the national per-brand
-- average. Includes Crash Rate % and Casualty Rate % to separate
-- volume effects (popular cars) from genuine danger signals.
-- ================================================================

WITH Cleaned_Complaints_Base AS (
    SELECT 
        REPLACE(UPPER(TRIM(MAKETXT)), '-', ' ') AS CLEAN_MAKE,
        DEATHS, INJURED, CRASH
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS
    WHERE MAKETXT IS NOT NULL AND PROD_TYPE = 'V' AND UPPER(TRIM(MAKETXT)) != 'UNKNOWN'
)
SELECT
    CLEAN_MAKE AS VEHICLE_MAKE,
    COUNT(*) AS TOTAL_COMPLAINTS,
    (
        SELECT ROUND(AVG(BRAND_VOLUME), 0)
        FROM (SELECT COUNT(*) AS BRAND_VOLUME FROM Cleaned_Complaints_Base GROUP BY CLEAN_MAKE)
    ) AS NATIONAL_BRAND_AVERAGE,
    SUM(DEATHS) AS TOTAL_DEATHS,
    SUM(INJURED) AS TOTAL_INJURIES,
    COUNT_IF(CRASH = 'Y') AS TOTAL_CRASHES,
    ROUND(COUNT_IF(CRASH = 'Y') * 100.0 / NULLIF(COUNT(*), 0), 2) AS CRASH_RATE_PCT,
    ROUND((SUM(DEATHS) + SUM(INJURED)) * 100.0 / NULLIF(COUNT(*), 0), 2) AS CASUALTY_RATE_PCT
FROM Cleaned_Complaints_Base
GROUP BY CLEAN_MAKE
HAVING COUNT(*) > (
    SELECT AVG(BRAND_VOLUME)
    FROM (SELECT COUNT(*) AS BRAND_VOLUME FROM Cleaned_Complaints_Base GROUP BY CLEAN_MAKE)
)
ORDER BY TOTAL_COMPLAINTS DESC;

In [ ]:
# ================================================================
# Query 14 Visualization: Volume vs Danger (Above-Average Brands)
# ================================================================

plt.style.use('dark_background')
AboveAvgComplaintBenchmark['TOTAL_COMPLAINTS'] = pd.to_numeric(AboveAvgComplaintBenchmark['TOTAL_COMPLAINTS'], errors='coerce')
AboveAvgComplaintBenchmark['CRASH_RATE_PCT'] = pd.to_numeric(AboveAvgComplaintBenchmark['CRASH_RATE_PCT'], errors='coerce')
AboveAvgComplaintBenchmark['CASUALTY_RATE_PCT'] = pd.to_numeric(AboveAvgComplaintBenchmark['CASUALTY_RATE_PCT'], errors='coerce')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('More Complaints = More Dangerous? Not Always.', fontsize=14, fontweight='bold')

df_sorted = AboveAvgComplaintBenchmark.sort_values('TOTAL_COMPLAINTS', ascending=True).tail(15)
ax1.barh(df_sorted['VEHICLE_MAKE'], df_sorted['TOTAL_COMPLAINTS'],
         color=plt.cm.Blues(np.linspace(0.3, 0.9, len(df_sorted))), edgecolor='white', linewidth=0.3)
ax1.set_xlabel('Total Complaints', fontsize=11, fontweight='bold')
ax1.set_title('Who Gets Complained About Above Average?', fontweight='bold')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

df_danger = AboveAvgComplaintBenchmark.nlargest(15, 'CRASH_RATE_PCT').sort_values('CRASH_RATE_PCT', ascending=True)
ax2.barh(df_danger['VEHICLE_MAKE'], df_danger['CRASH_RATE_PCT'],
         color=plt.cm.Reds(np.linspace(0.3, 0.9, len(df_danger))), edgecolor='white', linewidth=0.3)
ax2.set_xlabel('Crash Rate (%)', fontsize=11, fontweight='bold')
ax2.set_title('But Who Is Actually Dangerous Per Incident?', fontweight='bold')
ax2.grid(axis='x', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 14 - Volume vs Danger.svg', format='svg', bbox_inches='tight')

### Findings: Above-Average Consumer Complaint Benchmark

- **Is Tesla Actually Dangerous?** — With 21,926 complaints, a 12.65% Crash Rate, and 10.12% Casualty Rate, a Tesla incident is over twice as likely to result in a collision or casualty compared to Ford (~4.98%). Fewer cars on the road, but much higher per-incident danger
- **Does Popularity = Safety?** — No. Historical brands like GEO and SCION appear low on volume but their Crash Rates pierce 20%+. Volume represents market sales; rates isolate engineering danger. The HAVING subquery dynamically separates the two

## Query 15: They Failed the Crash Test — Then Got Recalled (EXISTS)

Extracts deduplicated recall campaigns targeting vehicles that scored ≤2 stars in NHTSA crash tests. Uses a correlated EXISTS subquery to link laboratory failures with subsequent real-world recall actions.

In [ ]:
%%sql -r LabFailureRecallMatching
-- ================================================================
-- Query 15: Laboratory Failure → Recall Matching (EXISTS)
-- ================================================================
-- Uses a correlated EXISTS subquery to find recall campaigns
-- targeting vehicles that scored ≤2 stars in NHTSA crash tests.
-- Applies CASE WHEN component normalization and campaign-level
-- MAX(POTAFF) deduplication for accurate vehicle counts.
-- ================================================================

WITH Deduplicated_Worst_Recalls AS (
    SELECT 
        UPPER(TRIM(r.MAKETXT)) AS VEHICLE_MAKE,
        UPPER(TRIM(r.MODELTXT)) AS VEHICLE_MODEL,
        TRY_CAST(REGEXP_REPLACE(r.YEARTXT, '[^0-9]', '') AS NUMBER) AS CLEAN_YEAR,
        UPPER(TRIM(r.CAMPNO)) AS UNIQUE_CAMPNO,
        r.POTAFF,
        CASE 
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('SERVICE BRAKES, HYDRAULIC', 'SERVICE BRAKES', 'SERVICE BRAKES, AIR', 'SERVICE BRAKES, ELECTRIC', 'PARKING BRAKE') THEN 'BRAKES & BRAKING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('ENGINE', 'ENGINE AND ENGINE COOLING') THEN 'ENGINE & COOLING'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('FUEL SYSTEM, GASOLINE', 'FUEL/PROPULSION SYSTEM', 'FUEL SYSTEM, OTHER', 'FUEL SYSTEM, DIESEL', 'HYBRID PROPULSION SYSTEM') THEN 'FUEL & PROPULSION SYSTEM'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('ELECTRONIC STABILITY CONTROL (ESC)', 'ELECTRONIC STABILITY CONTROL', 'TRACTION CONTROL SYSTEM') THEN 'ELECTRONIC STABILITY & TRACTION'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('VISIBILITY', 'VISIBILITY/WIPER') THEN 'VISIBILITY & WIPERS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('EXTERIOR LIGHTING', 'INTERIOR LIGHTING') OR UPPER(TRIM(r.COMPNAME)) LIKE '%LIGHTING%' THEN 'LIGHTING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('TIRES', 'WHEELS') THEN 'TIRES & WHEELS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('FORWARD COLLISION AVOIDANCE', 'LANE DEPARTURE', 'BACK OVER PREVENTION', 'COMMUNICATION') THEN 'ADAS & ADVANCED TECHNOLOGY'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('EQUIPMENT', 'EQUIPMENT ADAPTIVE/MOBILITY', 'TRAILER HITCHES') THEN 'EQUIPMENT & ACCESSORIES'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('CHILD SEAT', 'CHEST CLIP, BUCKLE, HARNESS', 'CARRY HANDLE, SHELL, BASE', 'TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)', 'INSERT, PADDING') THEN 'CHILD SEAT & RESTRAINT EQUIPMENT'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('UNKNOWN OR OTHER', 'OTHER/I AM NOT SURE', 'OTHER/UNKNOWN', 'NONE', '1', '120', 'FIRERELATED', '') OR r.COMPNAME IS NULL THEN 'OTHER / UNKNOWN'
            ELSE UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) 
        END AS CLEAN_COMPONENT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL r
    WHERE EXISTS (
        SELECT 1 
        FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY s
        WHERE UPPER(TRIM(s.MAKE)) = UPPER(TRIM(r.MAKETXT))
          AND UPPER(TRIM(s.MODEL)) = UPPER(TRIM(r.MODELTXT))
          AND TRY_CAST(s.MODEL_YR AS NUMBER) = TRY_CAST(REGEXP_REPLACE(r.YEARTXT, '[^0-9]', '') AS NUMBER)
          AND s.OVERALL_STARS <= 2
    )
    AND r.COMPNAME IS NOT NULL AND r.COMPNAME != '' AND r.RCLTYPECD != 'X'
),
Campaign_Isolated_Recalls AS (
    SELECT VEHICLE_MAKE, VEHICLE_MODEL, CLEAN_YEAR, CLEAN_COMPONENT, UNIQUE_CAMPNO, MAX(POTAFF) AS TRUE_AFFECTED_VEHICLES
    FROM Deduplicated_Worst_Recalls
    GROUP BY 1, 2, 3, 4, 5
)
SELECT 
    VEHICLE_MAKE, VEHICLE_MODEL, CLEAN_YEAR AS MODEL_YEAR, CLEAN_COMPONENT AS PRIMARY_COMPONENT,
    COUNT(DISTINCT UNIQUE_CAMPNO) AS RECALL_CAMPAIGNS_ISSUED,
    SUM(TRUE_AFFECTED_VEHICLES) AS TOTAL_VEHICLES_RECALLED
FROM Campaign_Isolated_Recalls
GROUP BY 1, 2, 3, 4
ORDER BY TOTAL_VEHICLES_RECALLED DESC
LIMIT 25;

In [ ]:
# ================================================================
# Query 15 Visualization: Low-Rated Vehicles with Recalls
# ================================================================

plt.style.use('dark_background')
LabFailureRecallMatching['TOTAL_VEHICLES_RECALLED'] = pd.to_numeric(LabFailureRecallMatching['TOTAL_VEHICLES_RECALLED'], errors='coerce')
LabFailureRecallMatching['RECALL_CAMPAIGNS_ISSUED'] = pd.to_numeric(LabFailureRecallMatching['RECALL_CAMPAIGNS_ISSUED'], errors='coerce')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('They Failed the Crash Test — Then Got Recalled', fontsize=14, fontweight='bold')

LabFailureRecallMatching['LABEL'] = LabFailureRecallMatching['VEHICLE_MAKE'] + ' ' + LabFailureRecallMatching['VEHICLE_MODEL'] + ' (' + LabFailureRecallMatching['MODEL_YEAR'].astype(str) + ')'

df_top = LabFailureRecallMatching.nlargest(15, 'TOTAL_VEHICLES_RECALLED').sort_values('TOTAL_VEHICLES_RECALLED', ascending=True)
ax1.barh(df_top['LABEL'], df_top['TOTAL_VEHICLES_RECALLED'] / 1e6,
         color=plt.cm.Reds(np.linspace(0.3, 0.9, 15)), edgecolor='white', linewidth=0.3)
ax1.set_xlabel('Vehicles Recalled (Millions)', fontsize=11, fontweight='bold')
ax1.set_title('Which Low-Rated Cars Got Hit Hardest?', fontweight='bold')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

comp_agg = LabFailureRecallMatching.groupby('PRIMARY_COMPONENT')['TOTAL_VEHICLES_RECALLED'].sum().nlargest(10).sort_values(ascending=True)
ax2.barh(comp_agg.index, comp_agg.values / 1e6, color=plt.cm.Oranges(np.linspace(0.3, 0.9, 10)), edgecolor='white', linewidth=0.3)
ax2.set_xlabel('Vehicles Recalled (Millions)', fontsize=11, fontweight='bold')
ax2.set_title('What Breaks on Unsafe Cars?', fontweight='bold')
ax2.grid(axis='x', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 15 - Failed Crash Test Then Recalled.svg', format='svg', bbox_inches='tight')

### Findings: Failed the Crash Test — Then Got Recalled

- **Do Shared Assembly Lines = Shared Risk?** — Yes. GMC Sierra 2500 (2013) and Chevrolet Silverado 2500 (2013) have identical recall footprints — 2 campaigns, 2.47M vehicles each. Same factory, same parts, same defects, different badge
- **What Happens After a Lab Failure?** — The Ford Ranger (2011) scored ≤2 stars in crash testing, then got hit with 5 airbag recalls impacting 5.27M vehicles. Poor lab performance is a leading indicator of future recall action
- **Which Component Dominates?** — AIR BAGS, overwhelmingly. Vehicles that fail crash tests AND get recalled are disproportionately recalled for airbag defects — confirming these two failure modes compound each other

## Query 16: The Hall of Shame — Worst-Rated Recalled Vehicles (EXISTS + CASE)

Pulls official recall campaigns targeting vehicles that failed crash tests, applying component normalization and campaign-level deduplication. These are the highest-risk configurations in the entire database.

In [ ]:
%%sql -r RecallsLowRatedVehicles
-- ================================================================
-- Query 16: Recalls for Low-Rated Vehicles (EXISTS + CASE)
-- ================================================================
-- Variant of Query 17 using TRY_CAST(YEARTXT AS NUMBER) directly
-- instead of REGEXP_REPLACE for year matching. Applies identical
-- component CASE WHEN mapping and campaign deduplication logic
-- to ensure structural parity across the analysis.
-- ================================================================

WITH Deduplicated_Worst_Recalls AS (
    SELECT 
        UPPER(TRIM(r.MAKETXT)) AS VEHICLE_MAKE,
        UPPER(TRIM(r.MODELTXT)) AS VEHICLE_MODEL,
        TRY_CAST(r.YEARTXT AS NUMBER) AS MODEL_YEAR,
        UPPER(TRIM(r.CAMPNO)) AS UNIQUE_CAMPNO,
        r.POTAFF,
        CASE 
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('SERVICE BRAKES, HYDRAULIC', 'SERVICE BRAKES', 'SERVICE BRAKES, AIR', 'SERVICE BRAKES, ELECTRIC', 'PARKING BRAKE') THEN 'BRAKES & BRAKING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('ENGINE', 'ENGINE AND ENGINE COOLING') THEN 'ENGINE & COOLING'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('FUEL SYSTEM, GASOLINE', 'FUEL/PROPULSION SYSTEM', 'FUEL SYSTEM, OTHER', 'FUEL SYSTEM, DIESEL', 'HYBRID PROPULSION SYSTEM') THEN 'FUEL & PROPULSION SYSTEM'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('ELECTRONIC STABILITY CONTROL (ESC)', 'ELECTRONIC STABILITY CONTROL', 'TRACTION CONTROL SYSTEM') THEN 'ELECTRONIC STABILITY & TRACTION'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('VISIBILITY', 'VISIBILITY/WIPER') THEN 'VISIBILITY & WIPERS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('EXTERIOR LIGHTING', 'INTERIOR LIGHTING') THEN 'LIGHTING SYSTEMS'
            WHEN UPPER(TRIM(r.COMPNAME)) LIKE '%LIGHTING%' THEN 'LIGHTING SYSTEMS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('TIRES', 'WHEELS') THEN 'TIRES & WHEELS'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('FORWARD COLLISION AVOIDANCE', 'LANE DEPARTURE', 'BACK OVER PREVENTION', 'COMMUNICATION') THEN 'ADAS & ADVANCED TECHNOLOGY'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('EQUIPMENT', 'EQUIPMENT ADAPTIVE/MOBILITY', 'TRAILER HITCHES') THEN 'EQUIPMENT & ACCESSORIES'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('CHILD SEAT', 'CHEST CLIP, BUCKLE, HARNESS', 'CARRY HANDLE, SHELL, BASE', 'TETHER, LOWER ANCHOR (ON CAR SEAT OR VEHICLE)', 'INSERT, PADDING') THEN 'CHILD SEAT & RESTRAINT EQUIPMENT'
            WHEN UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) IN ('UNKNOWN OR OTHER', 'OTHER/I AM NOT SURE', 'OTHER/UNKNOWN', 'NONE', '1', '120', 'FIRERELATED', '') OR r.COMPNAME IS NULL THEN 'OTHER / UNKNOWN'
            ELSE UPPER(TRIM(SPLIT_PART(r.COMPNAME, ':', 1))) 
        END AS CLEAN_COMPONENT
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL r
    WHERE EXISTS (
        SELECT 1 
        FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY s
        WHERE UPPER(TRIM(s.MAKE)) = UPPER(TRIM(r.MAKETXT))
          AND UPPER(TRIM(s.MODEL)) = UPPER(TRIM(r.MODELTXT))
          AND TRY_CAST(s.MODEL_YR AS NUMBER) = TRY_CAST(r.YEARTXT AS NUMBER)
          AND s.OVERALL_STARS <= 2
    )
    AND r.COMPNAME IS NOT NULL AND r.COMPNAME != '' AND r.RCLTYPECD != 'X'
),
Campaign_Isolated_Recalls AS (
    SELECT 
        VEHICLE_MAKE, VEHICLE_MODEL, MODEL_YEAR, CLEAN_COMPONENT, UNIQUE_CAMPNO,
        MAX(POTAFF) AS TRUE_AFFECTED_VEHICLES
    FROM Deduplicated_Worst_Recalls
    GROUP BY 1, 2, 3, 4, 5
)
SELECT 
    VEHICLE_MAKE, VEHICLE_MODEL, MODEL_YEAR,
    CLEAN_COMPONENT AS PRIMARY_COMPONENT,
    COUNT(DISTINCT UNIQUE_CAMPNO) AS RECALL_CAMPAIGNS_ISSUED,
    SUM(TRUE_AFFECTED_VEHICLES) AS TOTAL_VEHICLES_RECALLED
FROM Campaign_Isolated_Recalls
GROUP BY 1, 2, 3, 4
ORDER BY TOTAL_VEHICLES_RECALLED DESC
LIMIT 25;

In [ ]:
# ================================================================
# Query 16 Visualization: Worst-Rated Vehicles Requiring Recalls
# ================================================================

plt.style.use('dark_background')
RecallsLowRatedVehicles['TOTAL_VEHICLES_RECALLED'] = pd.to_numeric(RecallsLowRatedVehicles['TOTAL_VEHICLES_RECALLED'], errors='coerce')
RecallsLowRatedVehicles['RECALL_CAMPAIGNS_ISSUED'] = pd.to_numeric(RecallsLowRatedVehicles['RECALL_CAMPAIGNS_ISSUED'], errors='coerce')
RecallsLowRatedVehicles['MODEL_YEAR'] = pd.to_numeric(RecallsLowRatedVehicles['MODEL_YEAR'], errors='coerce').astype('Int64')

RecallsLowRatedVehicles['LABEL'] = (
    RecallsLowRatedVehicles['VEHICLE_MAKE'].astype(str) + ' ' +
    RecallsLowRatedVehicles['VEHICLE_MODEL'].astype(str) + ' (' +
    RecallsLowRatedVehicles['MODEL_YEAR'].astype(str) + ') — ' +
    RecallsLowRatedVehicles['PRIMARY_COMPONENT'].astype(str).str.title()
)

df_plot = RecallsLowRatedVehicles.dropna(subset=['TOTAL_VEHICLES_RECALLED']).nlargest(20, 'TOTAL_VEHICLES_RECALLED').sort_values('TOTAL_VEHICLES_RECALLED', ascending=True)

fig, ax = plt.subplots(figsize=(16, 9))

colors = plt.cm.YlOrRd(np.linspace(0.3, 0.95, len(df_plot)))
bars = ax.barh(df_plot['LABEL'], df_plot['TOTAL_VEHICLES_RECALLED'] / 1e6, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Vehicles Recalled (Millions)', fontsize=11, fontweight='bold')
ax.set_title('The Hall of Shame: Worst-Rated Cars That Needed Recalls', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.15, linestyle='--')
ax.bar_label(bars, labels=[f"{int(r['RECALL_CAMPAIGNS_ISSUED'])} campaigns" for _, r in df_plot.iterrows()],
             padding=3, fontsize=8, color='#e94560')
ax.set_xlim(0, df_plot['TOTAL_VEHICLES_RECALLED'].max() / 1e6 * 1.25)
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 16 - Hall of Shame Worst Rated Recalled.svg', format='svg', bbox_inches='tight')

### Findings: The Hall of Shame — Worst-Rated Recalled Vehicles

- **Who Lands in the Hall of Shame?** — Vehicles scoring ≤2 overall stars that also required safety recalls represent the worst-case scenario: known to be unsafe in crash testing AND confirmed defective in the real world
- **What's Breaking on the Worst Cars?** — The same vehicles reappear across multiple component categories — brakes, electrical systems, fuel systems — confirming these aren't isolated defects but chronically problematic platforms that manufacturers failed to fix across model years

## Query 17: Brand Safety Risk Tiers (CTE + CASE WHEN)

Segments brands into investment risk tiers based on the proportion of recalled volume subject to catastrophic "Do Not Drive" or "Park Outside" directives. The final risk classification answers: would you invest in this brand?

In [ ]:
%%sql -r BrandSafetyRiskTiers
-- ================================================================
-- Query 17: Brand Safety Portfolio Risk Tiers (CTE + CASE WHEN)
-- ================================================================
-- Deduplicates campaigns at the brand level using MAX(POTAFF),
-- calculates the proportion of each brand's recall volume subject
-- to catastrophic directives (Do Not Drive / Park Outside), then
-- applies a tiered CASE WHEN to classify brands into risk buckets
-- for portfolio management decisions.
-- ================================================================

WITH UniqueCampaigns AS (
    SELECT 
        REPLACE(UPPER(TRIM(MAKETXT)), '-', ' ') AS CLEAN_MAKE,
        UPPER(TRIM(CAMPNO)) AS CLEAN_CAMPNO,
        MAX(POTAFF) AS CAMPAIGN_AFFECTED_VEHICLES,
        MAX(CASE WHEN UPPER(TRIM(DO_NOT_DRIVE)) IN ('Y', 'YES', '1', 'TRUE') THEN 1 ELSE 0 END) AS IS_DND,
        MAX(CASE WHEN UPPER(TRIM(PARK_OUTSIDE)) IN ('Y', 'YES', '1', 'TRUE') THEN 1 ELSE 0 END) AS IS_PO
    FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL
    WHERE MAKETXT IS NOT NULL AND RCLTYPECD = 'V'
      AND UPPER(TRIM(MAKETXT)) NOT IN ('UNKNOWN', 'OTHER', 'ALL', '')
    GROUP BY 1, 2
),
RecallSeverity AS (
    SELECT 
        CLEAN_MAKE AS VEHICLE_MAKE,
        COUNT(DISTINCT CLEAN_CAMPNO) AS TOTAL_SAFETY_CAMPAIGNS,
        SUM(CAMPAIGN_AFFECTED_VEHICLES) AS TOTAL_VEHICLES_RECALLED,
        SUM(CASE WHEN IS_DND = 1 OR IS_PO = 1 THEN CAMPAIGN_AFFECTED_VEHICLES ELSE 0 END) AS SEVERE_VEHICLES_RECALLED
    FROM UniqueCampaigns
    GROUP BY CLEAN_MAKE
    HAVING SUM(CAMPAIGN_AFFECTED_VEHICLES) > 500000
)
SELECT 
    VEHICLE_MAKE, TOTAL_SAFETY_CAMPAIGNS, TOTAL_VEHICLES_RECALLED, SEVERE_VEHICLES_RECALLED,
    ROUND((SEVERE_VEHICLES_RECALLED * 100.0) / NULLIF(TOTAL_VEHICLES_RECALLED, 0), 2) AS SEVERE_RECALL_PCT,
    CASE 
        WHEN SEVERE_VEHICLES_RECALLED = 0 AND TOTAL_VEHICLES_RECALLED > 5000000 THEN 'LOW RISK (Software / OTA Dominant Profile)'
        WHEN (SEVERE_VEHICLES_RECALLED * 1.0 / NULLIF(TOTAL_VEHICLES_RECALLED, 0)) > 0.10 THEN 'HIGH RISK (Monitor Closely - Capital Intensive)'
        WHEN (SEVERE_VEHICLES_RECALLED * 1.0 / NULLIF(TOTAL_VEHICLES_RECALLED, 0)) > 0.02 THEN 'MODERATE RISK (Standard Warranty Outlay)'
        ELSE 'LOW RISK (Routine Maintenance Variance)'
    END AS INVESTMENT_RISK_TIER
FROM RecallSeverity
ORDER BY SEVERE_RECALL_PCT DESC;

In [ ]:
# ================================================================
# Query 17 Visualization: Brand Safety Risk Tiers
# ================================================================

plt.style.use('dark_background')
BrandSafetyRiskTiers['SEVERE_RECALL_PCT'] = pd.to_numeric(BrandSafetyRiskTiers['SEVERE_RECALL_PCT'], errors='coerce')
BrandSafetyRiskTiers['TOTAL_VEHICLES_RECALLED'] = pd.to_numeric(BrandSafetyRiskTiers['TOTAL_VEHICLES_RECALLED'], errors='coerce')

# ----------------------------------------------------------------
# Data Cleaning: Drop rows with NaN severity, limit to top 15 by
# severity for a readable chart
# ----------------------------------------------------------------
df_risk = BrandSafetyRiskTiers.dropna(subset=['SEVERE_RECALL_PCT']).copy()
df_risk['VEHICLE_MAKE'] = df_risk['VEHICLE_MAKE'].str[:20]

df_risk['RISK_TIER_CONSOLIDATED'] = df_risk['INVESTMENT_RISK_TIER'].apply(
    lambda x: 'LOW RISK' if 'LOW RISK' in x else x.split('(')[0].strip()
)

tier_colors = {
    'HIGH RISK (Monitor Closely - Capital Intensive)': '#e94560',
    'MODERATE RISK (Standard Warranty Outlay)': '#f5a623',
    'LOW RISK (Routine Maintenance Variance)': '#55efc4',
    'LOW RISK (Software / OTA Dominant Profile)': '#55efc4'
}

pie_tier_colors = {
    'HIGH RISK': '#e94560',
    'MODERATE RISK': '#f5a623',
    'LOW RISK': '#55efc4'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('Would You Invest in These Brands? A Safety Risk Card', fontsize=14, fontweight='bold')

# ----------------------------------------------------------------
# Left Plot: Top 15 brands ranked by severe recall %
# ----------------------------------------------------------------
df_sorted = df_risk.nlargest(15, 'SEVERE_RECALL_PCT').sort_values('SEVERE_RECALL_PCT', ascending=True)
bar_colors = [tier_colors.get(t, 'gray') for t in df_sorted['INVESTMENT_RISK_TIER']]
ax1.barh(df_sorted['VEHICLE_MAKE'], df_sorted['SEVERE_RECALL_PCT'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Severe Recall %', fontsize=11, fontweight='bold')
ax1.set_title('Who Has the Most Catastrophic Recalls?', fontweight='bold')
ax1.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='High Risk (>10%)')
ax1.axvline(x=2, color='orange', linestyle='--', alpha=0.5, label='Moderate Risk (>2%)')
ax1.legend(fontsize=8, loc='lower right')
ax1.grid(axis='x', alpha=0.15, linestyle='--')

# ----------------------------------------------------------------
# Right Plot: Pie chart of risk tier distribution (consolidated)
# ----------------------------------------------------------------
tier_counts = df_risk['RISK_TIER_CONSOLIDATED'].value_counts()
pie_colors = [pie_tier_colors.get(t, 'gray') for t in tier_counts.index]
ax2.pie(tier_counts.values, labels=tier_counts.index, colors=pie_colors,
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
ax2.set_title('How Many Brands Fall Into Each Risk Bucket?', fontweight='bold')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 17 - Brand Safety Risk Card.svg', format='svg', bbox_inches='tight')

### Findings: Brand Safety Portfolio Risk Tiers

- **Who's the Riskiest Investment?** — Mitsubishi leads at a staggering 48.9% of recall volume classified as catastrophic (Do Not Drive / Park Outside orders). Ram follows at 28%, then KIA at 22% — nearly half of every Mitsubishi recall carries a life-safety directive
- **Who Gets a Free Pass?** — Tesla, Honda, Acura, and Buick show zero severe physical recall directives. Their recalls are limited to software fixes or routine maintenance-level issues — the lowest-liability tier from an investment perspective
- **The Pie Tells the Story** — The majority of brands fall into the Low Risk bucket, meaning catastrophic recalls are concentrated in a handful of repeat offenders rather than spread evenly across the industry

---
# Part 7: Can We Predict What Happens Next?
*Using the relationships uncovered above, we build two predictive models: Linear Regression (can complaints + ratings predict injury severity?) and Logistic Regression (can complaint volume predict whether a major recall is coming?).*

## Query 18: Predicting Injuries — Linear Regression

Can complaint volume and safety star ratings predict how many injuries a vehicle will cause? Using the complaints-ratings JOIN as training data, we build a linear regression model to test this hypothesis.

In [ ]:
%%sql -r SafetyRatingsForRegression
-- ================================================================
-- Query 18: Complaints + Safety Ratings (for Linear Regression)
-- ================================================================
-- Joins complaints with safety ratings on make/model/year to
-- create a training dataset with complaint counts, injuries,
-- deaths, and star ratings per vehicle configuration.
-- ================================================================

SELECT
    c.MAKETXT AS MAKE,
    c.MODELTXT AS MODEL,
    c.YEARTXT AS MODEL_YEAR,
    COUNT(*) AS COMPLAINT_COUNT,
    SUM(c.DEATHS) AS TOTAL_DEATHS,
    SUM(c.INJURED) AS TOTAL_INJURIES,
    s.OVERALL_STARS,
    s.ROLLOVER_STARS
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS c
INNER JOIN BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_SAFETY s
    ON UPPER(c.MAKETXT) = UPPER(s.MAKE)
    AND UPPER(c.MODELTXT) = UPPER(s.MODEL)
    AND TRY_CAST(c.YEARTXT AS NUMBER) = s.MODEL_YR
GROUP BY c.MAKETXT, c.MODELTXT, c.YEARTXT, s.OVERALL_STARS, s.ROLLOVER_STARS
ORDER BY COMPLAINT_COUNT DESC;

In [ ]:
# ================================================================
# Query 18 Visualization: Linear Regression — Predicting Injuries
# ================================================================

df_cs_model = SafetyRatingsForRegression[['COMPLAINT_COUNT', 'OVERALL_STARS', 'ROLLOVER_STARS', 'TOTAL_INJURIES', 'TOTAL_DEATHS']].dropna()

X_cs_inj = df_cs_model[['COMPLAINT_COUNT', 'OVERALL_STARS', 'ROLLOVER_STARS']]
y_cs_inj = df_cs_model['TOTAL_INJURIES']

X_cs_inj_train, X_cs_inj_test, y_cs_inj_train, y_cs_inj_test = train_test_split(X_cs_inj, y_cs_inj, test_size=0.3, random_state=42)

lr_cs_inj = LinearRegression()
lr_cs_inj.fit(X_cs_inj_train, y_cs_inj_train)
y_cs_inj_pred = lr_cs_inj.predict(X_cs_inj_test)

print("MODEL: Linear Regression - Predicting Injuries from Crashes + Safety Ratings")
print("="*60)
print(f"Features: COMPLAINT_COUNT, OVERALL_STARS, ROLLOVER_STARS")
print(f"R² Score: {r2_score(y_cs_inj_test, y_cs_inj_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_cs_inj_test, y_cs_inj_pred)):.2f}")
print(f"MAE: {mean_absolute_error(y_cs_inj_test, y_cs_inj_pred):.2f}")
print(f"\nCoefficients:")
for feat, coef in zip(X_cs_inj.columns, lr_cs_inj.coef_):
    print(f"  {feat}: {coef:.4f}")
print(f"  Intercept: {lr_cs_inj.intercept_:.4f}")

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Can We Predict Injuries from Complaints & Safety Stars?', fontsize=14, fontweight='bold')

axes[0].scatter(y_cs_inj_test, y_cs_inj_pred, alpha=0.7, color='#00d2d3', edgecolors='white', linewidths=0.5, s=50)
axes[0].plot([y_cs_inj_test.min(), y_cs_inj_test.max()], [y_cs_inj_test.min(), y_cs_inj_test.max()], '--', color='#ffd700', lw=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Injuries')
axes[0].set_ylabel('Predicted Injuries')
axes[0].set_title('How Close Are Our Predictions?', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.15, linestyle='--')

residuals_inj = y_cs_inj_test - y_cs_inj_pred
axes[1].hist(residuals_inj, bins=8, color='#e94560', edgecolor='white', alpha=0.85)
axes[1].axvline(x=0, color='#ffd700', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residuals (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Where Does the Model Go Wrong?', fontweight='bold')
axes[1].grid(axis='y', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 18 - Linear Regression Injuries.svg', format='svg', bbox_inches='tight')

### Interpretation: Linear Regression — Predicting Injuries from Complaints + Safety Ratings

- **How Close Are Our Predictions?** — The model does well on the easy cases (low-injury vehicles cluster tightly along the diagonal), but falls apart at the extremes. High-injury outliers scatter wildly from the prediction line — these are the crashes where factors we can't measure (speed, road conditions, driver behavior) dominate
- **Where Does the Model Go Wrong?** — Residuals are centered at zero, so at least it's not systematically biased. But the spread tells you this model captures direction, not precision. Complaint count is doing most of the heavy lifting; safety stars barely move the needle since most vehicles score 4–5 anyway

## Query 19: Predicting Major Recalls — Logistic Regression

The final question: can a spike in complaints predict whether a major recall is coming? A logistic regression model classifies make/component combinations as high-recall vs low-recall based on complaint volume, injuries, and deaths.

In [ ]:
%%sql -r ComplaintsRecallsForLogistic
-- ================================================================
-- Query 19: Complaints Linked to Recalls (for Logistic Regression)
-- ================================================================
-- Full dataset of complaints joined to recalls without LIMIT,
-- providing complete training data for the logistic regression
-- model. Each row is a make with complaint volume, recall
-- campaigns, deaths, and injuries.
-- ================================================================

SELECT
    c.MAKETXT AS MAKE,
    COUNT(DISTINCT c.CMPLID) AS COMPLAINT_COUNT,
    COUNT(DISTINCT r.CAMPNO) AS RELATED_RECALLS,
    SUM(c.DEATHS) AS DEATHS_FROM_COMPLAINTS,
    SUM(c.INJURED) AS INJURIES_FROM_COMPLAINTS
FROM BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_COMPLAINTS c
LEFT JOIN BUSN32120_NHTSA_PROJECT.PUBLIC.NHTSA_RECALL r
    ON UPPER(c.MAKETXT) = UPPER(r.MAKETXT)
    AND UPPER(c.MODELTXT) = UPPER(r.MODELTXT)
    AND c.YEARTXT = r.YEARTXT
WHERE c.MAKETXT IS NOT NULL
    AND UPPER(TRIM(c.MAKETXT)) NOT IN ('UNKNOWN', 'OTHER', '')
GROUP BY c.MAKETXT
HAVING COUNT(DISTINCT c.CMPLID) > 10
ORDER BY COMPLAINT_COUNT DESC;

In [ ]:
# ================================================================
# Query 19 Visualization: Logistic Regression — Predicting Recalls
# ================================================================

df_recall_model = ComplaintsRecallsForLogistic[['COMPLAINT_COUNT', 'INJURIES_FROM_COMPLAINTS', 'DEATHS_FROM_COMPLAINTS', 'RELATED_RECALLS']].dropna()
df_recall_model = df_recall_model.apply(pd.to_numeric, errors='coerce').dropna()

median_recalls = df_recall_model['RELATED_RECALLS'].median()
df_recall_model['HIGH_RECALLS'] = (df_recall_model['RELATED_RECALLS'] > median_recalls).astype(int)

X_recall = df_recall_model[['COMPLAINT_COUNT', 'INJURIES_FROM_COMPLAINTS', 'DEATHS_FROM_COMPLAINTS']]
y_recall = df_recall_model['HIGH_RECALLS']
X_recall_train, X_recall_test, y_recall_train, y_recall_test = train_test_split(X_recall, y_recall, test_size=0.3, random_state=42)

log_recall_model = LogisticRegression(max_iter=1000, random_state=42)
log_recall_model.fit(X_recall_train, y_recall_train)
y_recall_pred = log_recall_model.predict(X_recall_test)

print("MODEL: Logistic Regression - Predicting High Recall Volume")
print("="*60)
print(f'Accuracy: {accuracy_score(y_recall_test, y_recall_pred):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_recall_test, y_recall_pred, target_names=['Low Recalls', 'High Recalls']))

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Can Complaints Predict Whether a Recall Will Be Major?', fontsize=14, fontweight='bold')

cm_recall = confusion_matrix(y_recall_test, y_recall_pred)
group_names = ['True Negative', 'False Positive', 'False Negative', 'True Positive']
group_counts = [f'{v}' for v in cm_recall.flatten()]
group_pcts = [f'{v:.1%}' for v in cm_recall.flatten() / cm_recall.sum()]
labels = [f'{n}\n{c}\n({p})' for n, c, p in zip(group_names, group_counts, group_pcts)]
labels = np.array(labels).reshape(2, 2)

im = axes[0].imshow(cm_recall, cmap='RdYlGn', alpha=0.85)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, labels[i, j], ha='center', va='center', fontsize=11, color='black', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Low Recalls', 'High Recalls'])
axes[0].set_yticklabels(['Low Recalls', 'High Recalls'])
axes[0].set_xlabel('Predicted', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Actual', fontsize=11, fontweight='bold')
axes[0].set_title('How Often Is the Model Right?', fontweight='bold')

feature_names_log = ['Complaint Count', 'Injuries', 'Deaths']
coefs = log_recall_model.coef_[0]
colors_log = ['#e94560' if c < 0 else '#55efc4' for c in coefs]
axes[1].barh(feature_names_log, coefs, color=colors_log, edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='white', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('Coefficient (Log-Odds)', fontsize=11, fontweight='bold')
axes[1].set_title('What Drives the Prediction?', fontweight='bold')
axes[1].grid(axis='x', alpha=0.15, linestyle='--')

plt.tight_layout()
if SVG_SAVE: fig.savefig('Plots/Query 19 - Logistic Regression Recalls.svg', format='svg', bbox_inches='tight')

### Interpretation: Logistic Regression — Predicting High Recall Volume from Complaints

- **How Often Is the Model Right?** — The confusion matrix lays it out: diagonal cells are correct predictions, off-diagonal are mistakes. The model's strength is identifying low-recall cases correctly (True Negatives), but it's more likely to miss a high-recall case than to falsely flag a low one
- **What Drives the Prediction?** — Death count dominates. Its coefficient is the longest bar by far, confirming what regulators already suspect: a spike in deaths is the clearest early warning that a major recall is coming. Injuries and complaint add signal, but volume alone gets you most of the way there

# Conclusion

## Summary of Findings

This analysis traced the full lifecycle of vehicle safety defects across four NHTSA datasets — crash test ratings, consumer complaints, investigations, and recalls — and revealed several key insights:

### 1. The Safety Floor Is Rising, But the Ceiling Is Fixed
Crash test ratings have converged across major manufacturers, with 79% of year-over-year transitions showing no change. Achieving 5 stars is now table stakes for top brands, yet no body style averages a perfect score across all categories. The industry is safer than ever, but incremental improvements are increasingly difficult.

### 2. Complaint Volume Reflects Market Share, Not Engineering Failure
Ford, Chevrolet, and Dodge dominate complaint rankings because they dominate U.S. roads. However, **per-incident danger rates** tell a different story: Tesla's 12.65% crash rate and GEO/SCION's 20%+ rates reveal that smaller-volume brands can carry disproportionate risk per complaint filed.

### 3. Safety Ratings Do Not Predict Non-Crash Complaints
The JOIN between complaints and ratings confirmed that a 5-star vehicle with a faulty transmission will still generate thousands of complaints. Crash test performance predicts crash survivability — not long-term mechanical reliability. Consumers must evaluate both dimensions independently.

### 4. Complaint Spikes Are the Clearest Recall Predictor
The logistic regression model confirmed what regulators already suspect: complaint volume is the dominant predictor of whether a major recall is coming. Injuries and deaths add marginal signal, but sheer volume gets you most of the way there.

### 5. Shared Platforms Mean Shared Risk
GMC Sierra and Chevrolet Silverado appear with identical recall footprints — same factory, same parts, same defects, different badge. Consumers researching one model should always check its platform siblings.

### 6. Some Brands Carry Structural Safety Liability
KIA's 22.46% catastrophic recall rate (engine fires, "Do Not Drive" orders) versus Tesla/Honda/Acura's 0% severe physical recall rate represents a measurable, investable difference in brand-level safety risk.

---

## Limitations

- **Predictive models showed weak explanatory power** (R² near zero for linear regression), indicating that complaints and star ratings alone cannot predict injury counts — unmeasured factors like road conditions, driver behavior, and vehicle age dominate outcomes
- **Complaint data has survivorship bias** — only drivers who know about and choose to file complaints appear in the dataset
- **Safety ratings test new vehicles only** — degradation over time is not captured in NCAP scores

---

## Implications for Car Buyers

| Decision Factor | What This Analysis Shows |
|---|---|
| Crash test ratings | Necessary but not sufficient — check all four categories, not just the overall star count |
| Complaint history | Look at per-incident rates (crash %, casualty %), not raw complaint counts |
| Recall history | Check platform siblings, not just your exact model |
| Brand risk tier | Avoid brands with high catastrophic recall percentages unless you accept the liability |
| Safety technology | Brands with OTA-updatable systems carry lower long-term recall risk |

---

## Future Work

- **Normalizing data using vehicle volumes sold will bring insight into the data.
- **For emerging brands more data is required. Continue monitoring.
- **Consider exploring global safety data.

---

*Analysis conducted using NHTSA public data. SQL techniques include JOINs, Window Functions, GROUP BY CUBE/ROLLUP, CTEs, EXISTS subqueries, CASE WHEN classifications, and HAVING benchmarks. Predictive modeling used Linear and Logistic Regression via scikit-learn.No recommendations for or against any brands, makes or models are the intention of this data analysis.*

# Appendix: Analysis that did not add value

In [ ]:
# ================================================================
# Model Fail: Linear Regression — Predicting Deaths
# ================================================================

X_cs_death = ComplaintsWithSafetyRatings[['COMPLAINT_COUNT', 'OVERALL_STARS', 'ROLLOVER_STARS']]
y_cs_death = ComplaintsWithSafetyRatings['TOTAL_DEATHS']

cs_death_data = pd.concat([X_cs_death, y_cs_death], axis=1).dropna()
X_cs_death = cs_death_data[['COMPLAINT_COUNT', 'OVERALL_STARS', 'ROLLOVER_STARS']]
y_cs_death = cs_death_data['TOTAL_DEATHS']

X_cs_death_train, X_cs_death_test, y_cs_death_train, y_cs_death_test = train_test_split(X_cs_death, y_cs_death, test_size=0.3, random_state=42)

lr_cs_death = LinearRegression()
lr_cs_death.fit(X_cs_death_train, y_cs_death_train)
y_cs_death_pred = lr_cs_death.predict(X_cs_death_test)

print("MODEL 4: Linear Regression - Predicting Deaths from Crashes + Safety Ratings")
print("="*60)
print(f"Features: COMPLAINT_COUNT, OVERALL_STARS, ROLLOVER_STARS")
print(f"R² Score: {r2_score(y_cs_death_test, y_cs_death_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_cs_death_test, y_cs_death_pred)):.2f}")
print(f"MAE: {mean_absolute_error(y_cs_death_test, y_cs_death_pred):.2f}")
print(f"\nCoefficients:")
for feat, coef in zip(X_cs_death.columns, lr_cs_death.coef_):
    print(f"  {feat}: {coef:.6f}")
print(f"  Intercept: {lr_cs_death.intercept_:.4f}")

In [ ]:
# ================================================================
# Model Fail: Logistic  Regression — Predicting High Low Deaths
# ================================================================

df_model2 = RankingMakesByComplaints[['TOTAL_COMPLAINTS', 'CRASH_COUNT', 'FIRE_COUNT', 'TOTAL_DEATHS', 'TOTAL_INJURIES']].dropna()

median_deaths = df_model2['TOTAL_DEATHS'].median()
df_model2['HIGH_DEATH_RATE'] = (df_model2['TOTAL_DEATHS'] > median_deaths).astype(int)

X2 = df_model2[['TOTAL_COMPLAINTS', 'CRASH_COUNT', 'FIRE_COUNT', 'TOTAL_INJURIES']]
y2 = df_model2['HIGH_DEATH_RATE']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42)

log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X2_train, y2_train)
y2_pred = log_model.predict(X2_test)

print("MODEL 2: Logistic Regression - Predicting High vs Low Death Rate")
print("="*60)
print(f"Accuracy: {accuracy_score(y2_test, y2_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y2_test, y2_pred, target_names=['Low Deaths', 'High Deaths']))